# 🥒 Zucchini CNN — Incremental Ablation Study
**Baseline → + Color Features → + Dual Head → + Composite Loss**

| Step | What is added | Goal |
|------|---------------|------|
| **M0 — Baseline** | Simple custom CNN, MSE loss, single regression head | Reference |
| **M1 — + Color Features** | 54-dim LAB handcrafted feature branch fused at FC | Does global color help? |
| **M2 — + Dual Head** | Ordinal head (7 thresholds) on top of M1 | Does ordinal structure help? |
| **M3 — + Composite Loss** | L_ord + L_huber + L_rank on top of M2 | Does richer loss help? |

**Settings (shared for all models)**
- Images: `nocc` cropped, 128×128, CIELAB input
- CV: 5-fold stratified, augmentation only on train folds
- Early stopping: patience=30 | LR reduction: patience=8
- Monitoring: R², RMSE, Acc@0, Acc@1, Spearman ρ, Kendall τ
- Frieze: per-score line, max 15 images per row


## 0 · Imports & Setup

In [ ]:
import os, glob, random, warnings, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import tifffile as tiff
import cv2
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from scipy.stats import spearmanr, kendalltau, skew, kurtosis
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (confusion_matrix, r2_score,
                             mean_squared_error, mean_absolute_error)

import torch
import torch.nn as nn  # noqa
import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if device.type == 'cuda':
    torch.cuda.manual_seed(SEED)
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

os.makedirs('training/model', exist_ok=True)
os.makedirs('training/figures', exist_ok=True)

def savefig(name, dpi=150):
    path = f'training/figures/{name}'
    plt.savefig(path, dpi=dpi, bbox_inches='tight')
    print(f'  💾 {path}')


## 1 · Configuration

In [ ]:
# ── EDIT THIS PATH ────────────────────────────────────────────────────────────
NOCC_ROOT = r'F:\Aymen\server\statistical_filtering'
# ─────────────────────────────────────────────────────────────────────────────

IMG_SIZE    = (128, 128)    # 128×128 nocc cropped images
BATCH_SIZE  = 16
EPOCHS      = 300
K_FOLDS     = 5
PATIENCE_ES = 30            # early stopping patience
PATIENCE_LR = 8             # LR reduction patience
LR_INIT     = 5e-4
LR_MIN      = 1e-6
LR_FACTOR   = 0.5

SCORE_FOLDERS = list(range(1, 10))   # folders 1-9 → scores 1-9
NUM_CLASSES   = 9
IDX_TO_SCORE  = np.arange(1, 10, dtype=np.float32)   # 1..9
FEAT_DIM      = 54                   # 54-dim LAB feature vector
MAX_PER_ROW   = 15                   # frieze: max images per score line

print(f'Image size  : {IMG_SIZE}')
print(f'K-folds     : {K_FOLDS}')
print(f'Max epochs  : {EPOCHS}')
print(f'ES patience : {PATIENCE_ES}')
print(f'LR patience : {PATIENCE_LR}')


## 2 · Load Images → RGB + LAB

In [ ]:
def load_tif_rgb(path, size):
    img = tiff.imread(path)
    if img.ndim == 4: img = img[0]
    if img.ndim == 3 and img.shape[0] in (1,3,4):
        img = np.transpose(img, (1,2,0))
    if img.ndim == 2:        img = np.stack([img]*3, axis=-1)
    elif img.shape[-1] == 1: img = np.concatenate([img]*3, axis=-1)
    elif img.shape[-1] == 4: img = img[...,:3]
    if img.dtype != np.uint8:
        img = img.astype(np.float32)
        lo, hi = img.min(), img.max()
        img = (img - lo) / (hi - lo + 1e-8)
        img = (img * 255).astype(np.uint8)
    return cv2.resize(img, (size[1], size[0]), interpolation=cv2.INTER_AREA)

def rgb_to_lab(img_rgb):
    return cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)

def lab_to_rgb(img_lab):
    return cv2.cvtColor(img_lab, cv2.COLOR_LAB2RGB)


def collect_data(root, score_folders, img_size):
    """
    Walk each Score=NN (or plain NN) subfolder under root.
    Accepts nocc images named with any of:
      +nocc+, +NOCC+, _nocc_, nocc (case-insensitive substring match).
    Falls back to ALL .tif files in the folder if none match.
    Prints a per-folder diagnostic so you can see exactly what was found.
    """
    rgb_list, lab_list, label_list = [], [], []

    print(f"\nSearching root: {root}")
    if not os.path.exists(root):
        raise FileNotFoundError(f"NOCC_ROOT not found: {root}")

    # List what's actually inside root (helps debug)
    top_items = sorted(os.listdir(root))
    print(f"  Contents ({len(top_items)} items): {top_items[:20]}")

    for folder_num in score_folders:
        # Try several common folder naming conventions
        candidates = [
            os.path.join(root, str(folder_num)),
            os.path.join(root, f"Score={folder_num:02d}"),
            os.path.join(root, f"Score={folder_num}"),
            os.path.join(root, f"{folder_num:02d}"),
        ]
        folder_path = next((p for p in candidates if os.path.isdir(p)), None)
        if folder_path is None:
            print(f"  [Score {folder_num}] ⚠  No matching folder found (tried {candidates})")
            continue

        # Collect all tif files
        all_tifs = (glob.glob(os.path.join(folder_path, "*.tif"))
                  + glob.glob(os.path.join(folder_path, "*.TIF"))
                  + glob.glob(os.path.join(folder_path, "*.tiff")))
        all_tifs = list(set(all_tifs))

        # Filter to nocc files (case-insensitive)
        nocc_tifs = [p for p in all_tifs
                     if "nocc" in os.path.basename(p).lower()]

        # Use nocc if found, else fall back to everything
        chosen = nocc_tifs if nocc_tifs else all_tifs
        tag = "nocc" if nocc_tifs else "ALL (no nocc found — using fallback)"

        print(f"  [Score {folder_num}] folder={os.path.basename(folder_path)} "
              f"| all_tifs={len(all_tifs)} | {tag}={len(chosen)}")

        for p in chosen:
            try:
                rgb = load_tif_rgb(p, img_size)
                lab = rgb_to_lab(rgb)
                rgb_list.append(rgb)
                lab_list.append(lab)
                label_list.append(folder_num - 1)   # 0-indexed
            except Exception as e:
                print(f"    ⚠ Failed {os.path.basename(p)}: {e}")

    if len(rgb_list) == 0:
        raise RuntimeError(
            "No images loaded!\n"
            f"Check that NOCC_ROOT is correct: {root}\n"
            "And that subfolders exist (e.g. 1/, 2/, ... or Score=01/, ...)"
        )

    print(f"\n  ✔ Total loaded: {len(rgb_list)} images "
          f"across {len(set(label_list))} score classes\n")
    return (np.array(rgb_list,  dtype=np.uint8),
            np.array(lab_list,  dtype=np.uint8),
            np.array(label_list, dtype=np.int32))


X_rgb, X_lab, y_raw = collect_data(NOCC_ROOT, SCORE_FOLDERS, IMG_SIZE)

# Distribution plot
unique, counts = np.unique(y_raw, return_counts=True)
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(unique + 1, counts, color='steelblue', edgecolor='black')
ax.set_xlabel('Score'); ax.set_ylabel('Count')
ax.set_title('Image Distribution per Score Class')
ax.set_xticks(unique + 1)
for u, c in zip(unique, counts):
    ax.text(u + 1, c + 0.3, str(c), ha='center', fontsize=9)
plt.tight_layout(); savefig('class_distribution.png'); plt.show()


## 3 · 54-dim LAB Handcrafted Features
Per LAB channel: mean (1) + std (1) + 16-bin histogram (16) = 18 dims × 3 channels = **54 total**


In [ ]:
def extract_lab_features(img_lab_uint8):
    """
    Returns a 54-dim float32 feature vector from a LAB uint8 image.

    NOTE on why M1 showed ~zero gain over M0:
    The original version of this function only computed whole-image mean/std/
    histogram per channel. A CNN that ends in global-average-pooling can
    approximate exactly those statistics from its own learned filters, so the
    handcrafted branch was largely redundant information, not new information.
    To make the feature branch worth fusing, it should capture things GAP
    discards: spatial structure (center vs. border), color-space relationships
    across channels (chroma, hue), and distribution *shape* (skew/kurtosis),
    not just marginal mean/std -- a heavily-lopsided or heavy-tailed color
    distribution (e.g. a small blemished patch) can share the same mean/std as
    a uniform one but is a very different quality signal.

    The 54 dims are fully allocated (no dead zero-padding):

      L,a,b marginal mean + std        (3ch × 2)         =  6 dims
      L,a,b marginal median            (3ch × 1)         =  3 dims  (NEW)
      L,a,b marginal skewness          (3ch × 1)         =  3 dims  (NEW — shape)
      L,a,b marginal excess kurtosis   (3ch × 1)         =  3 dims  (NEW — shape)
      L,a,b 8-bin histograms           (8 × 3ch)         = 24 dims
      center-vs-border mean delta      (3ch)             =  3 dims  (spatial)
      chroma C* = sqrt(a*²+b*²) mean/std                  =  2 dims  (cross-channel)
      hue angle h = atan2(b*,a*) mean/std (circular)       =  2 dims  (cross-channel)
      a*b* correlation coefficient                        =  1 dim  (cross-channel)
      a*, b* interquartile range (IQR)                     =  2 dims  (NEW — a*/b* spread)
      quadrant means (4 spatial quadrants, L channel)      =  4 dims  (spatial)
      edge-region L std (border ring, 10px)                =  1 dim  (spatial)
      ---------------------------------------------------------
      total                                                = 54 dims
    """
    h, w = img_lab_uint8.shape[:2]
    L = img_lab_uint8[:, :, 0].astype(np.float32)
    A_raw = img_lab_uint8[:, :, 1].astype(np.float32)          # raw a*, uint8 space
    B_raw = img_lab_uint8[:, :, 2].astype(np.float32)          # raw b*, uint8 space
    A = A_raw - 128.0                                          # center a* around 0
    B = B_raw - 128.0                                          # center b* around 0

    feats = []
    channels = (L, A_raw, B_raw)   # L, a*, b* in their native uint8 range

    # ── per-channel marginal mean/std (6 dims) ───────────────────────────────
    for ch in channels:
        feats.append(ch.mean() / 255.0)
        feats.append(ch.std()  / 255.0)

    # ── per-channel marginal median (3 dims) ─────────────────────────────────
    for ch in channels:
        feats.append(float(np.median(ch)) / 255.0)

    # ── per-channel marginal skewness (3 dims) ───────────────────────────────
    # Fisher-Pearson skewness; 0 = symmetric, >0 = right-tailed, <0 = left-tailed.
    for ch in channels:
        feats.append(float(np.nan_to_num(skew(ch.flatten()))))

    # ── per-channel marginal excess kurtosis (3 dims) ────────────────────────
    # Fisher definition (normal → 0); >0 = heavy-tailed/peaked, <0 = flat.
    for ch in channels:
        feats.append(float(np.nan_to_num(kurtosis(ch.flatten()))))

    # ── per-channel 8-bin histograms (24 dims) ───────────────────────────────
    for ch in channels:
        hist, _ = np.histogram(ch, bins=8, range=(0, 256))
        hist = hist.astype(np.float32)
        hist /= (hist.sum() + 1e-8)
        feats.extend(hist.tolist())

    # ── spatial: center vs border mean delta, per channel (3 dims) ──────────
    cy0, cy1 = h // 4, 3 * h // 4
    cx0, cx1 = w // 4, 3 * w // 4
    border_mask = np.ones((h, w), dtype=bool)
    border_mask[cy0:cy1, cx0:cx1] = False
    for ch in channels:
        center_mean = ch[cy0:cy1, cx0:cx1].mean()
        border_mean = ch[border_mask].mean()
        feats.append((center_mean - border_mean) / 255.0)

    # ── cross-channel: chroma C* = sqrt(a*^2+b*^2), mean & std (2 dims) ─────
    chroma = np.sqrt(A**2 + B**2)
    feats.append(chroma.mean() / 180.0)
    feats.append(chroma.std()  / 180.0)

    # ── cross-channel: hue angle (circular mean/std via vector averaging) ───
    hue = np.arctan2(B, A)
    sin_mean, cos_mean = np.sin(hue).mean(), np.cos(hue).mean()
    circ_mean = np.arctan2(sin_mean, cos_mean)
    circ_var  = 1.0 - np.sqrt(sin_mean**2 + cos_mean**2)   # 0=concentrated, 1=spread
    feats.append((circ_mean + np.pi) / (2*np.pi))
    feats.append(circ_var)

    # ── cross-channel: a*/b* correlation (1 dim) ─────────────────────────────
    if A.std() > 1e-6 and B.std() > 1e-6:
        corr = np.corrcoef(A.flatten(), B.flatten())[0, 1]
    else:
        corr = 0.0
    feats.append(float(np.nan_to_num(corr)))

    # ── a*, b* interquartile range -- dedicated spread measure per channel ──
    # (mean/std/histogram already describe location & shape; IQR adds a
    # robust, outlier-resistant spread specifically for the two chrominance
    # channels the person cares most about for greenness/color-quality.)
    a_p75, a_p25 = np.percentile(A, [75, 25])
    b_p75, b_p25 = np.percentile(B, [75, 25])
    feats.append(float(a_p75 - a_p25) / 180.0)
    feats.append(float(b_p75 - b_p25) / 180.0)

    # ── spatial: 4-quadrant mean of L channel (4 dims) ───────────────────────
    hh, hw = h // 2, w // 2
    quads = [L[:hh, :hw], L[:hh, hw:], L[hh:, :hw], L[hh:, hw:]]
    for q in quads:
        feats.append(q.mean() / 255.0)

    # ── spatial: outer-ring std of L (texture near border, 1 dim) ───────────
    ring_w = max(int(min(h, w) * 0.08), 2)
    ring_mask = np.zeros((h, w), dtype=bool)
    ring_mask[:ring_w, :] = True; ring_mask[-ring_w:, :] = True
    ring_mask[:, :ring_w] = True; ring_mask[:, -ring_w:] = True
    feats.append(L[ring_mask].std() / 255.0)

    feats = np.array(feats, dtype=np.float32)   # 54 dims, fully populated

    # defensive pad/crop in case FEAT_DIM is ever changed upstream
    if len(feats) < FEAT_DIM:
        feats = np.concatenate([feats, np.zeros(FEAT_DIM - len(feats), dtype=np.float32)])
    return feats[:FEAT_DIM]


feats_raw = np.array([extract_lab_features(lab) for lab in X_lab], dtype=np.float32)
print(f"Feature matrix : {feats_raw.shape}   (N × {FEAT_DIM})")
print("  6 mean/std + 3 median + 3 skew + 3 kurtosis + 24 histogram "
      "+ 3 center/border + 2 chroma + 2 hue + 1 corr + 2 a*/b* IQR "
      "+ 4 quadrant + 1 edge-texture = 54 active dims (no padding)")


## 4 · Augmentation Functions (geometry on RGB, photometric on LAB)

In [ ]:
def aug_flip(rgb, v):
    return cv2.flip(rgb, [1, 0, -1][v])

def aug_rotate(rgb, v):
    angles = [15, -15, 30]
    h, w   = rgb.shape[:2]
    M = cv2.getRotationMatrix2D((w/2, h/2), angles[v], 1.0)
    return cv2.warpAffine(rgb, M, (w,h), borderMode=cv2.BORDER_REFLECT)

def aug_zoom(rgb, v):
    pcts = [0.10, 0.20, 0.30]; f = pcts[v]
    h, w = rgb.shape[:2]; dh, dw = int(h*f/2), int(w*f/2)
    return cv2.resize(rgb[dh:h-dh, dw:w-dw], (w,h), interpolation=cv2.INTER_LINEAR)

def aug_blur(lab, v):
    return cv2.GaussianBlur(lab, (3,3), [0.5,1.0,1.5][v])

def aug_noise(lab, v):
    stds = [3,6,9]
    n = np.random.normal(0, stds[v], lab.shape).astype(np.float32)
    return np.clip(lab.astype(np.float32)+n, 0, 255).astype(np.uint8)

AUG_CATALOGUE = [
    ('Flip',     aug_flip,   'rgb', 3),
    ('Rotation', aug_rotate, 'rgb', 3),
    ('Zoom',     aug_zoom,   'rgb', 3),
    ('Blur',     aug_blur,   'lab', 3),
    ('Noise',    aug_noise,  'lab', 3),
]
print(f'Augmentation factor : ×{1 + sum(n for *_,n in AUG_CATALOGUE)}')


def augment_fold(train_idx):
    aug_lab, aug_feat, aug_y = [], [], []
    for i in train_idx:
        aug_lab.append(X_lab[i]); aug_feat.append(feats_raw[i]); aug_y.append(y_raw[i])
    for aug_name, aug_fn, domain, n_var in AUG_CATALOGUE:
        for v in range(n_var):
            for i in train_idx:
                if domain == 'rgb':
                    alab = rgb_to_lab(aug_fn(X_rgb[i], v))
                else:
                    alab = aug_fn(X_lab[i], v)
                aug_lab.append(alab)
                aug_feat.append(extract_lab_features(alab))
                aug_y.append(y_raw[i])
    return (np.array(aug_lab, dtype=np.uint8),
            np.array(aug_feat, dtype=np.float32),
            np.array(aug_y, dtype=np.int32))


## 5 · PyTorch Dataset

In [ ]:
class ZucchiniDataset(Dataset):
    """LAB image + optional handcrafted features, regression target = score 1..9."""
    def __init__(self, lab_imgs, feats, labels, augment=False, use_feats=True):
        # Normalize LAB → [0,1]
        self.imgs   = torch.from_numpy(lab_imgs.astype(np.float32) / 255.0).permute(0,3,1,2)
        self.feats  = torch.from_numpy(feats.astype(np.float32)) if use_feats else None
        self.scores = torch.from_numpy(IDX_TO_SCORE[labels]).unsqueeze(1)  # (N,1)
        self.labels = labels
        self.augment = augment
        self.use_feats = use_feats

    def __len__(self): return len(self.imgs)

    def __getitem__(self, idx):
        img   = self.imgs[idx].clone()
        score = self.scores[idx]
        if self.augment and random.random() < 0.3:
            if random.random() < 0.5: img = img.flip(2)    # H flip
            if random.random() < 0.5: img = img.flip(1)    # V flip
        if self.use_feats:
            return img, self.feats[idx], score, self.labels[idx]
        else:
            return img, score, self.labels[idx]


## 6 · Model Architectures

Four models share the same CNN backbone (4 conv blocks, 128→64→32→16 spatial reduction):

| Model | Feature branch | Head | Loss |
|-------|---------------|------|------|
| **M0** Baseline | None | Single regression | MSELoss |
| **M1** + Color Feats | 54→64→32 MLP fused after GAP | Single regression | MSELoss |
| **M2** + Dual Head | Same as M1 | Ordinal (7 thresholds) + regression | MSELoss on reg |
| **M3** + Composite Loss | Same as M2 | Same as M2 | L_ord + L_huber + L_rank |


In [ ]:
# ── Shared CNN backbone ────────────────────────────────────────────────────────
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.25):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(dropout),
        )
    def forward(self, x): return self.block(x)


class CNNBackbone(nn.Module):
    """4-block CNN: 128→64→32→16→GAP(256)"""
    def __init__(self):
        super().__init__()
        self.blocks = nn.Sequential(
            ConvBlock(3,  32, 0.15),   # 128→64
            ConvBlock(32, 64, 0.20),   # 64→32
            ConvBlock(64,128, 0.25),   # 32→16
            ConvBlock(128,256,0.30),   # 16→8
        )
        self.gap = nn.AdaptiveAvgPool2d(1)

    def forward(self, x):
        return self.gap(self.blocks(x)).flatten(1)   # (B, 256)

CNN_DIM = 256   # output dim of backbone


# ── M0 : Baseline ─────────────────────────────────────────────────────────────
class M0_Baseline(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = CNNBackbone()
        self.fc = nn.Sequential(
            nn.Linear(CNN_DIM, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.35),
            nn.Linear(128, 1)
        )
    def forward(self, img, feat=None):
        return self.fc(self.backbone(img))   # (B,1) regression output


# ── M1 : + Color Features (late fusion) ──────────────────────────────────────
class M1_ColorFeats(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = CNNBackbone()
        self.feat_mlp = nn.Sequential(
            nn.Linear(FEAT_DIM, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
        )
        fused_dim = CNN_DIM + 32
        self.fc = nn.Sequential(
            nn.Linear(fused_dim, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.35),
            nn.Linear(128, 1)
        )
    def forward(self, img, feat):
        cnn_out = self.backbone(img)
        h       = self.feat_mlp(feat)
        fused   = torch.cat([cnn_out, h], dim=1)
        return self.fc(fused)   # (B,1)


# ── M2 : + Dual Head (ordinal + regression) ───────────────────────────────────
NUM_ORD = 8   # 8 thresholds for scores 1..9
class M2_DualHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = CNNBackbone()
        self.feat_mlp = nn.Sequential(
            nn.Linear(FEAT_DIM, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
        )
        fused_dim = CNN_DIM + 32
        self.shared_fc = nn.Sequential(
            nn.Linear(fused_dim, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.35),
        )
        self.ord_head = nn.Linear(128, NUM_ORD)    # ordinal logits
        self.reg_head = nn.Linear(128, 1)          # regression output
    def forward(self, img, feat):
        cnn_out = self.backbone(img)
        h       = self.feat_mlp(feat)
        fused   = self.shared_fc(torch.cat([cnn_out, h], dim=1))
        return self.ord_head(fused), self.reg_head(fused)   # (B,8), (B,1)

    @staticmethod
    def predict_score(ord_logits, reg_out):
        """Ensemble: 0.7 × ordinal + 0.3 × regression."""
        ord_score = 1.0 + torch.sigmoid(ord_logits).sum(dim=1, keepdim=True)
        reg_score = reg_out.clamp(1.0, 9.0)
        return 0.7 * ord_score + 0.3 * reg_score


# ── M3 : + Composite Loss (same arch as M2) ──────────────────────────────────
class M3_CompositeLoss(nn.Module):
    """Same architecture as M2 — the change is in the loss function."""
    def __init__(self):
        super().__init__()
        self.backbone = CNNBackbone()
        self.feat_mlp = nn.Sequential(
            nn.Linear(FEAT_DIM, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
        )
        fused_dim = CNN_DIM + 32
        self.shared_fc = nn.Sequential(
            nn.Linear(fused_dim, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.35),
        )
        self.ord_head = nn.Linear(128, NUM_ORD)
        self.reg_head = nn.Linear(128, 1)
    def forward(self, img, feat):
        cnn_out = self.backbone(img)
        h       = self.feat_mlp(feat)
        fused   = self.shared_fc(torch.cat([cnn_out, h], dim=1))
        return self.ord_head(fused), self.reg_head(fused)

    @staticmethod
    def predict_score(ord_logits, reg_out):
        ord_score = 1.0 + torch.sigmoid(ord_logits).sum(dim=1, keepdim=True)
        reg_score = reg_out.clamp(1.0, 9.0)
        return 0.7 * ord_score + 0.3 * reg_score


# ── Print parameter counts ────────────────────────────────────────────────────
for name, cls in [('M0 Baseline', M0_Baseline),
                   ('M1 +ColorFeats', M1_ColorFeats),
                   ('M2 +DualHead', M2_DualHead),
                   ('M3 +CompLoss', M3_CompositeLoss)]:
    m = cls()
    n = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'{name:20s}: {n:>10,} trainable params')


## 7 · Loss Functions

| Model | Loss |
|-------|------|
| M0, M1 | MSELoss |
| M2 | MSELoss on regression head only |
| M3 | 1.0 × L_ord + 0.5 × L_huber + 0.5 × L_rank |


In [ ]:
def loss_mse(pred, target):
    """Single head MSE: pred=(B,1), target=(B,1)."""
    return nn.functional.mse_loss(pred, target)

def loss_ordinal_bce(y_score, ord_logits):
    """
    Ordinal BCE on NUM_ORD cumulative thresholds.
    y_score : (B,)  — raw float scores (1..9)
    ord_logits: (B, NUM_ORD)
    y_ord[i,k] = 1  if  score_i > (k+1)
    """
    thresholds = torch.arange(1, NUM_ORD + 1,
                               dtype=torch.float32,
                               device=y_score.device).unsqueeze(0)   # (1, NUM_ORD)
    y_ord = (y_score.unsqueeze(1) > thresholds).float()              # (B, NUM_ORD)
    return nn.functional.binary_cross_entropy_with_logits(ord_logits, y_ord)

def loss_dual_mse(ord_logits, reg_out, target):
    """
    M2: regression MSE + ordinal BCE.

    FIX: the previous version only computed MSE on the regression head and
    completely ignored ord_logits, so the ordinal head never received any
    gradient during training — at inference its (effectively random) output
    was still blended into the final prediction via 0.7*ord_score+0.3*reg_score,
    which is why M2 underperformed M0/M1. Both heads must be trained for a
    dual-head model to make sense.
    """
    y_score = target.squeeze(1)
    l_reg = nn.functional.mse_loss(reg_out, target)
    l_ord = loss_ordinal_bce(y_score, ord_logits)
    return l_reg + l_ord

def loss_huber(reg_out, target, delta=1.0):
    return nn.functional.huber_loss(reg_out, target, delta=delta)

def loss_ranking(reg_out, target, margin=1.0):
    pred = reg_out.squeeze(1)           # (B,)
    t    = target.squeeze(1)            # (B,)
    s_i  = t.unsqueeze(1)              # (B,1)
    s_j  = t.unsqueeze(0)              # (1,B)
    p_i  = pred.unsqueeze(1)           # (B,1)
    p_j  = pred.unsqueeze(0)           # (1,B)
    gap  = s_i - s_j                   # (B,B)
    mask = (gap.abs() >= 1.0).float()
    l    = torch.clamp(margin - gap.sign() * (p_i - p_j), min=0.0)
    return (mask * l).sum() / (mask.sum() + 1e-8)

def loss_composite(ord_logits, reg_out, target):
    """
    M3 composite loss.
    ord_logits : (B, NUM_ORD)
    reg_out    : (B, 1)
    target     : (B, 1)  — scores 1..9
    """
    y_score = target.squeeze(1)        # (B,) — needed by ordinal & ranking losses
    l_ord   = loss_ordinal_bce(y_score, ord_logits)
    l_hub   = loss_huber(reg_out, target)
    l_rank  = loss_ranking(reg_out, target)
    return 1.0 * l_ord + 0.5 * l_hub + 0.5 * l_rank

print("Loss functions defined ✔")
print(f"  NUM_ORD = {NUM_ORD}  (thresholds at scores 1..{NUM_ORD})")
print("  M2 loss now trains BOTH heads (regression MSE + ordinal BCE)")


## 8 · Evaluation Metrics

In [ ]:
def within_acc(preds, targets, gap):
    return float(np.mean(np.abs(np.round(preds) - targets) <= gap))

def compute_metrics(preds, targets):
    preds   = np.array(preds, dtype=np.float32).flatten()
    targets = np.array(targets, dtype=np.float32).flatten()
    r2   = r2_score(targets, preds)
    mse  = mean_squared_error(targets, preds)
    rmse = float(np.sqrt(mse))
    rho, p_rho = spearmanr(targets, preds)
    tau, p_tau = kendalltau(targets, preds)
    acc0 = within_acc(preds, targets, 0)
    acc1 = within_acc(preds, targets, 1)
    return dict(r2=r2, rmse=rmse, rho=float(rho), tau=float(tau),
                acc0=acc0, acc1=acc1)

print('Metrics helper defined ✔')


## 9 · 5-Fold Stratified Cross-Validation Splits

In [ ]:
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
fold_splits = list(skf.split(np.arange(len(y_raw)), y_raw))

print(f'StratifiedKFold — {K_FOLDS} folds over {len(y_raw)} images')
for fi,(tr,va) in enumerate(fold_splits):
    cnt = np.bincount(y_raw[va], minlength=NUM_CLASSES)
    print(f'  Fold {fi+1}: train={len(tr):5d}  val={len(va):4d}  '
          f'val class dist={cnt.tolist()}')


## 10 · Generic Training Engine

In [ ]:
def make_loaders(lab_aug, feat_aug, y_aug,
                 lab_val, feat_val, y_val,
                 batch_size):
    """Always include features in dataset; M0 simply ignores feat in forward()."""
    ds_tr = ZucchiniDataset(lab_aug, feat_aug, y_aug, augment=True,  use_feats=True)
    ds_va = ZucchiniDataset(lab_val, feat_val, y_val, augment=False, use_feats=True)
    tr = DataLoader(ds_tr, batch_size=batch_size, shuffle=True,  num_workers=0, pin_memory=False)
    va = DataLoader(ds_va, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=False)
    return tr, va


def get_preds_and_targets(model, loader, model_id, device):
    """Run inference on a DataLoader; returns (preds, scores, labels) as numpy."""
    model.eval()
    preds_list, scores_list, labels_list = [], [], []
    with torch.no_grad():
        for img, feat, score, lbl in loader:
            img   = img.to(device)
            feat  = feat.to(device)
            score = score.to(device)

            if model_id == 'M0':
                out  = model(img)            # feat ignored by M0
                pred = out                   # (B,1)
            elif model_id == 'M1':
                out  = model(img, feat)
                pred = out                   # (B,1)
            elif model_id == 'M3pure':       # composite-style loss, single head, no color
                out  = model(img, feat)      # feat ignored by this arch
                pred = out                   # (B,1)
            else:                            # M2, M3, M4, M23, M2pure -- dual-head models
                ord_l, reg_o = model(img, feat)
                pred = model.predict_score(ord_l, reg_o)  # (B,1)

            preds_list.append(pred.cpu().float())
            scores_list.append(score.cpu().float())
            labels_list.append(torch.tensor(lbl).cpu())

    preds  = torch.cat(preds_list).numpy().flatten()
    scores = torch.cat(scores_list).numpy().flatten()
    labels = torch.cat(labels_list).numpy().flatten().astype(int)
    return preds, scores, labels


def train_one_fold(model_cls, model_id, fold_idx,
                   tr_orig_idx, va_orig_idx, loss_fn=None):
    """Augment train fold, train model, collect OOF predictions."""
    # ── Augment train, keep val clean ────────────────────────────────────────
    lab_aug_f, feat_aug_f, y_aug_f = augment_fold(tr_orig_idx)
    lab_val_f  = X_lab[va_orig_idx]
    feat_val_f = feats_raw[va_orig_idx]
    y_val_f    = y_raw[va_orig_idx]

    print(f'  Train (aug)={len(lab_aug_f):6d}  |  Val={len(lab_val_f):4d}')

    tr_loader, va_loader = make_loaders(
        lab_aug_f, feat_aug_f, y_aug_f,
        lab_val_f, feat_val_f, y_val_f,
        batch_size=BATCH_SIZE)

    # ── Fresh model & optimizer ──────────────────────────────────────────────
    torch.manual_seed(SEED + fold_idx)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED + fold_idx)
    model = model_cls().to(device)
    optimizer = optim.Adam(model.parameters(), lr=LR_INIT, weight_decay=1e-4)

    hist = {k: [] for k in ['train_loss', 'val_r2', 'val_rmse',
                              'val_rho', 'val_tau', 'val_acc0', 'val_acc1', 'lr']}

    best_r2      = -np.inf
    wait_es      = 0
    wait_lr      = 0
    best_epoch   = 0
    best_path    = f'training/model/best_{model_id}_fold{fold_idx+1}.pt'

    t0 = time.time()
    for epoch in range(EPOCHS):
        # ── Train one epoch ──────────────────────────────────────────────────
        model.train()
        ep_loss = []
        for img, feat, score, _ in tr_loader:
            img   = img.to(device)
            feat  = feat.to(device)
            score = score.to(device)
            optimizer.zero_grad()

            if model_id == 'M0':
                out  = model(img)
                loss = loss_mse(out, score)
            elif model_id == 'M1':
                out  = model(img, feat)
                loss = loss_mse(out, score)
            elif model_id == 'M2':
                ord_l, reg_o = model(img, feat)
                loss = loss_dual_mse(ord_l, reg_o, score)
            elif model_id == 'M3pure':   # composite-style loss (huber+rank), single head, no color
                out  = model(img, feat)  # feat ignored by this arch
                _single_fn = loss_fn if loss_fn is not None else loss_mse
                loss = _single_fn(out, score)
            else:   # M3, M4, M23, M2pure, and any new dual-head + composite-loss composition
                ord_l, reg_o = model(img, feat)
                # loss_fn lets a caller pin a *specific* composite-loss weighting
                # (e.g. the original vanilla 1.0/0.5/0.5 weights, or loss_dual_mse
                # for a "dual head only" arm) instead of silently picking up
                # whatever `loss_composite` was last rebound to by the Optuna
                # tuning cells further down the notebook.
                _composite_fn = loss_fn if loss_fn is not None else loss_composite
                loss = _composite_fn(ord_l, reg_o, score)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            ep_loss.append(loss.item())

        tr_loss = float(np.mean(ep_loss))

        # ── Validate ─────────────────────────────────────────────────────────
        val_preds, val_scores, _ = get_preds_and_targets(model, va_loader, model_id, device)
        m = compute_metrics(val_preds, val_scores)
        cur_lr = optimizer.param_groups[0]['lr']

        hist['train_loss'].append(tr_loss)
        hist['val_r2'].append(m['r2']);    hist['val_rmse'].append(m['rmse'])
        hist['val_rho'].append(m['rho']);  hist['val_tau'].append(m['tau'])
        hist['val_acc0'].append(m['acc0']); hist['val_acc1'].append(m['acc1'])
        hist['lr'].append(cur_lr)

        # ── Checkpoint on R² ─────────────────────────────────────────────────
        marker = ''
        if m['r2'] > best_r2:
            best_r2    = m['r2']
            best_epoch = epoch
            wait_es    = 0
            wait_lr    = 0
            torch.save(model.state_dict(), best_path)
            marker = f'  ✅ BEST R²={best_r2:.4f}'
        else:
            wait_es += 1
            wait_lr += 1

        # ── LR reduction ─────────────────────────────────────────────────────
        if wait_lr >= PATIENCE_LR:
            new_lr = max(cur_lr * LR_FACTOR, LR_MIN)
            for pg in optimizer.param_groups:
                pg['lr'] = new_lr
            wait_lr = 0
            marker += f'  📉 LR→{new_lr:.1e}'

        if (epoch + 1) % 10 == 0 or marker:
            print(f'  Ep {epoch+1:3d} | loss={tr_loss:.4f} '
                  f'R²={m["r2"]:.3f} RMSE={m["rmse"]:.3f} '
                  f'ρ={m["rho"]:.3f} τ={m["tau"]:.3f} '
                  f'Acc@0={m["acc0"]*100:.0f}% Acc@1={m["acc1"]*100:.0f}%{marker}')

        # ── Early stopping ───────────────────────────────────────────────────
        if wait_es >= PATIENCE_ES:
            print(f'  ⚠ Early stop @ epoch {epoch+1}  '
                  f'(best epoch={best_epoch+1}  best R²={best_r2:.4f})')
            break

    elapsed = (time.time() - t0) / 60
    print(f'  Fold done in {elapsed:.1f} min')

    # ── OOF predictions from best checkpoint ─────────────────────────────────
    model.load_state_dict(torch.load(best_path, map_location=device))
    oof_p, oof_t, oof_l = get_preds_and_targets(model, va_loader, model_id, device)
    return oof_p, oof_t, oof_l, hist, best_epoch


print('Training engine ready ✔')


## 11 · Run All Four Models (5-Fold CV Each)
> **This is the main training cell.** Expect ≈ 30–90 min total (GPU dependent).


In [ ]:
MODEL_REGISTRY = {
    'M0': M0_Baseline,
    'M1': M1_ColorFeats,
    'M2': M2_DualHead,
    'M3': M3_CompositeLoss,
}
MODEL_LABELS = {
    'M0': 'M0 — Baseline',
    'M1': 'M1 — +Color Feats',
    'M2': 'M2 — +Dual Head',
    'M3': 'M3 — +Composite Loss',
}
model_ids = list(MODEL_LABELS.keys())

ALL_RESULTS = {}

for model_id, model_cls in MODEL_REGISTRY.items():
    print('\n' + '='*70)
    print(f'  {MODEL_LABELS[model_id]}'.center(70))
    print('='*70)

    oof_pred_all  = np.full(len(y_raw), np.nan, dtype=np.float32)
    oof_true_all  = IDX_TO_SCORE[y_raw]
    oof_label_all = y_raw.copy()
    fold_hists        = []
    fold_metrics_list = []

    t0 = time.time()
    for fold_idx, (tr_idx, va_idx) in enumerate(fold_splits):
        print(f'\n  ── Fold {fold_idx+1}/{K_FOLDS} ────────────────────────────')
        oof_p, oof_t, oof_l, hist, best_ep = train_one_fold(
            model_cls, model_id, fold_idx, tr_idx, va_idx)

        oof_pred_all[va_idx] = oof_p
        fold_hists.append(hist)
        m = compute_metrics(oof_p, oof_t)
        fold_metrics_list.append({**m, 'fold': fold_idx+1, 'best_epoch': best_ep+1})

    elapsed = (time.time() - t0) / 60
    oof_metrics = compute_metrics(oof_pred_all, oof_true_all)
    print(f'\n  ✔ {model_id} done in {elapsed:.1f} min')
    print(f'  OOF R²={oof_metrics["r2"]:.4f}  RMSE={oof_metrics["rmse"]:.4f}  '
          f'ρ={oof_metrics["rho"]:.4f}  Acc@0={oof_metrics["acc0"]*100:.1f}%  '
          f'Acc@1={oof_metrics["acc1"]*100:.1f}%')

    ALL_RESULTS[model_id] = {
        'oof_pred':     oof_pred_all,
        'oof_true':     oof_true_all,
        'oof_label':    oof_label_all,
        'fold_hist':    fold_hists,
        'fold_metrics': fold_metrics_list,
        'oof_metrics':  oof_metrics,
    }

print('\n' + '='*70)
print('  ALL MODELS COMPLETE'.center(70))
print('='*70)


## 12 · Global Comparison Table

In [ ]:
rows = []
for mid, label in MODEL_LABELS.items():
    m = ALL_RESULTS[mid]['oof_metrics']
    fm = pd.DataFrame(ALL_RESULTS[mid]['fold_metrics'])
    rows.append({
        'Model': label,
        'R² (OOF)':      f"{m['r2']:.4f}",
        'RMSE (OOF)':    f"{m['rmse']:.4f}",
        'Spearman ρ':    f"{m['rho']:.4f}",
        'Kendall τ':     f"{m['tau']:.4f}",
        'Acc@0 (%)':     f"{m['acc0']*100:.1f}",
        'Acc@1 (%)':     f"{m['acc1']*100:.1f}",
        'R² (mean±std)': f"{fm['r2'].mean():.4f}±{fm['r2'].std():.4f}",
    })

df_cmp = pd.DataFrame(rows)
print('\n' + '='*90)
print('  COMPARISON TABLE — OOF Metrics across 5 Folds')
print('='*90)
print(df_cmp.to_string(index=False))
print('='*90)
df_cmp.to_csv('training/comparison_table.csv', index=False)
print('  Saved → training/comparison_table.csv')


## 12b · Tuning the Ordinal→Score Formula (M2 / M3)

The dual-head models currently combine outputs with hand-picked weights:

```python
ord_score = 1.0 + sigmoid(ord_logits).sum(dim=1)
final     = 0.7 * ord_score + 0.3 * reg_score
```

This is arbitrary. We now search over **three competing aggregation formulas** and their
coefficients using Optuna, optimizing directly for **OOF R²** (computed on the already-trained
fold checkpoints — no retraining needed, this only changes how we turn the two raw head
outputs into a final score).

**Formula candidates**

| Name | Formula | Idea |
|------|---------|------|
| `weighted_sum` | `w·ord_score + (1-w)·reg_score` | current approach, tune `w` only |
| `expected_value` | `Σ k·P(score=k)` from softmax over class logits | proper expected value, no independent-sigmoid inconsistency |
| `cumulative_rank` | `1 + Σ clip(sigmoid(logit_k), 0,1)` with **monotonic enforcement** (cumulative max) | fixes non-monotonic ordinal probabilities before summing |

We also jointly tune the composite-loss weights `(λ_ord, λ_huber, λ_rank)` *if* you choose to
retrain M3 with the winning weights afterward (separate optional cell below).


In [ ]:
import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Re-collect raw ordinal logits + regression outputs for M2 and M3 ─────────
# We need RAW per-sample (ord_logits, reg_out, true_score) triples per fold,
# not just the already-combined OOF predictions, so we can re-combine them
# under different formulas without retraining.

def collect_raw_heads(model_cls, model_id):
    """Reload each fold's best checkpoint and collect raw (ord_logits, reg_out, true_score)."""
    all_ord, all_reg, all_true, all_idx = [], [], [], []
    for fold_idx, (tr_idx, va_idx) in enumerate(fold_splits):
        ckpt_path = f'training/model/best_{model_id}_fold{fold_idx+1}.pt'
        model = model_cls().to(device)
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
        model.eval()

        lab_val  = X_lab[va_idx]
        feat_val = feats_raw[va_idx]
        y_val    = y_raw[va_idx]
        ds = ZucchiniDataset(lab_val, feat_val, y_val, augment=False, use_feats=True)
        dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False)

        with torch.no_grad():
            for img, feat, score, lbl in dl:
                img, feat = img.to(device), feat.to(device)
                ord_l, reg_o = model(img, feat)
                all_ord.append(ord_l.cpu().numpy())
                all_reg.append(reg_o.cpu().numpy())
                all_true.append(score.numpy())
        all_idx.extend(va_idx.tolist())

    return (np.concatenate(all_ord), np.concatenate(all_reg).flatten(),
            np.concatenate(all_true).flatten(), np.array(all_idx))

print('Collecting raw head outputs for M2 ...')
raw_ord_M2, raw_reg_M2, raw_true_M2, raw_idx_M2 = collect_raw_heads(M2_DualHead, 'M2')
print(f'  {raw_ord_M2.shape[0]} samples, ord_logits shape={raw_ord_M2.shape}')

print('Collecting raw head outputs for M3 ...')
raw_ord_M3, raw_reg_M3, raw_true_M3, raw_idx_M3 = collect_raw_heads(M3_CompositeLoss, 'M3')
print(f'  {raw_ord_M3.shape[0]} samples, ord_logits shape={raw_ord_M3.shape}')


In [ ]:
def sigmoid_np(x):
    return 1.0 / (1.0 + np.exp(-x))

def softmax_np(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)


def formula_weighted_sum(ord_logits, reg_out, w):
    """Current approach: w * ordinal_score + (1-w) * regression_score."""
    ord_score = 1.0 + sigmoid_np(ord_logits).sum(axis=1)
    reg_score = np.clip(reg_out, 1.0, 9.0)
    return w * ord_score + (1.0 - w) * reg_score


def formula_cumulative_rank(ord_logits, reg_out, w):
    """
    Fix non-monotonicity: enforce P(score>k) is non-increasing in k
    via a running minimum (cumulative max of violations removed),
    then sum — this prevents inconsistent thresholds from inflating the score.
    """
    probs = sigmoid_np(ord_logits)                      # (N, NUM_ORD)
    # enforce monotonic non-increasing probabilities left-to-right
    probs_mono = np.minimum.accumulate(probs, axis=1)
    ord_score = 1.0 + probs_mono.sum(axis=1)
    reg_score = np.clip(reg_out, 1.0, 9.0)
    return w * ord_score + (1.0 - w) * reg_score


def formula_expected_value(ord_logits, reg_out, w):
    """
    Convert ordinal logits into per-class probabilities via adjacent
    differences of cumulative sigmoid probs, then take expected value.
    P(score=k) = P(score>k-1) - P(score>k)   (clipped at 0)
    """
    probs = sigmoid_np(ord_logits)                       # (N, NUM_ORD), P(score>1..8)
    probs_mono = np.minimum.accumulate(probs, axis=1)     # enforce monotonic
    cum = np.concatenate([np.ones((probs_mono.shape[0],1)), probs_mono,
                          np.zeros((probs_mono.shape[0],1))], axis=1)   # P(score>0)=1, P(score>9)=0
    class_probs = np.clip(cum[:, :-1] - cum[:, 1:], 0, None)            # (N, 9) for scores 1..9
    class_probs = class_probs / (class_probs.sum(axis=1, keepdims=True) + 1e-8)
    scores = np.arange(1, 10, dtype=np.float64)
    exp_val = (class_probs * scores).sum(axis=1)
    reg_score = np.clip(reg_out, 1.0, 9.0)
    return w * exp_val + (1.0 - w) * reg_score


FORMULAS = {
    'weighted_sum':     formula_weighted_sum,
    'cumulative_rank':  formula_cumulative_rank,
    'expected_value':   formula_expected_value,
}
print('3 candidate formulas defined:', list(FORMULAS.keys()))


In [ ]:
def closed_form_optimal_w(ord_logits, reg_out, true_score):
    """Exact R²-maximizing weight for w*ord_score + (1-w)*reg_score, no search needed."""
    ord_score = 1.0 + sigmoid_np(ord_logits).sum(axis=1)
    reg_score = np.clip(reg_out, 1.0, 9.0)

    d = ord_score - reg_score
    r = true_score - reg_score
    w_unconstrained = float(np.sum(d * r) / (np.sum(d ** 2) + 1e-8))
    w_star = float(np.clip(w_unconstrained, 0.0, 1.0))

    pred = w_star * ord_score + (1 - w_star) * reg_score
    metrics = compute_metrics(pred, true_score)
    return w_star, w_unconstrained, metrics


for model_label, raw_ord, raw_reg, raw_true in [
    ('M2', raw_ord_M2, raw_reg_M2, raw_true_M2),
    ('M3', raw_ord_M3, raw_reg_M3, raw_true_M3),
]:
    w_star, w_unclipped, m = closed_form_optimal_w(raw_ord, raw_reg, raw_true)
    base_pred = formula_weighted_sum(raw_ord, raw_reg, 0.7)
    base_m = compute_metrics(base_pred, raw_true)

    print(f'{model_label}:')
    print(f'  Closed-form optimum : w = {w_star:.4f}  (unclipped = {w_unclipped:.4f})')
    print(f'  -> {w_star:.3f}·ord_score + {1-w_star:.3f}·reg_score')
    print(f'  R² @ w=0.7 (current): {base_m["r2"]:.4f}')
    print(f'  R² @ w={w_star:.3f} (tuned): {m["r2"]:.4f}   (Δ = {m["r2"]-base_m["r2"]:+.4f})')
    print()

### Optuna search — best formula + weight per model (M2, M3)

In [ ]:
def run_optuna_search(ord_logits, reg_out, true_score, model_label, n_trials=150):
    def objective(trial):
        formula_name = trial.suggest_categorical('formula', list(FORMULAS.keys()))
        w = trial.suggest_float('w', 0.0, 1.0)
        pred = FORMULAS[formula_name](ord_logits, reg_out, w)
        r2 = r2_score(true_score, pred)
        return r2   # maximize

    study = optuna.create_study(direction='maximize',
                                sampler=TPESampler(seed=SEED))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    best = study.best_params
    best_pred = FORMULAS[best['formula']](ord_logits, reg_out, best['w'])
    best_metrics = compute_metrics(best_pred, true_score)

    # baseline (current hardcoded 0.7/0.3 weighted_sum) for comparison
    base_pred = formula_weighted_sum(ord_logits, reg_out, 0.7)
    base_metrics = compute_metrics(base_pred, true_score)

    print(f'\n{"="*65}')
    print(f'  {model_label} — Optuna search ({n_trials} trials)')
    print(f'{"="*65}')
    print(f'  BEFORE (hardcoded w=0.7, weighted_sum):')
    print(f'    R²={base_metrics["r2"]:.4f}  RMSE={base_metrics["rmse"]:.4f}  '
          f'ρ={base_metrics["rho"]:.4f}  Acc@1={base_metrics["acc1"]*100:.1f}%')
    print(f'  AFTER  (tuned formula=\'{best["formula"]}\', w={best["w"]:.3f}):')
    print(f'    R²={best_metrics["r2"]:.4f}  RMSE={best_metrics["rmse"]:.4f}  '
          f'ρ={best_metrics["rho"]:.4f}  Acc@1={best_metrics["acc1"]*100:.1f}%')
    dr2 = best_metrics['r2'] - base_metrics['r2']
    print(f'  ΔR² = {dr2:+.4f}')
    print(f'{"="*65}')

    return study, best, best_pred, best_metrics, base_metrics


study_M2, best_params_M2, tuned_pred_M2, tuned_metrics_M2, base_metrics_M2 = \
    run_optuna_search(raw_ord_M2, raw_reg_M2, raw_true_M2, 'M2 — Dual Head', n_trials=150)

study_M3, best_params_M3, tuned_pred_M3, tuned_metrics_M3, base_metrics_M3 = \
    run_optuna_search(raw_ord_M3, raw_reg_M3, raw_true_M3, 'M3 — Composite Loss', n_trials=150)


In [ ]:
# ── Optimization history + formula comparison plots ──────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

for col, (study, label) in enumerate([(study_M2, 'M2'), (study_M3, 'M3')]):
    trials_df = study.trials_dataframe()

    # R² over trials
    ax = axes[0, col]
    ax.plot(trials_df['number'], trials_df['value'], 'o', alpha=0.3, ms=3, color='steelblue')
    running_best = trials_df['value'].cummax()
    ax.plot(trials_df['number'], running_best, color='crimson', lw=2, label='Best so far')
    ax.set_xlabel('Trial'); ax.set_ylabel('R²')
    ax.set_title(f'{label} — Optuna Search Progress', fontweight='bold')
    ax.legend(); ax.grid(alpha=0.3)

    # R² by formula type (boxplot)
    ax = axes[1, col]
    formula_groups = [trials_df[trials_df['params_formula']==f]['value'].values
                      for f in FORMULAS.keys()]
    bp = ax.boxplot(formula_groups, labels=list(FORMULAS.keys()), patch_artist=True)
    for patch, c in zip(bp['boxes'], ['#4e79a7','#f28e2b','#59a14f']):
        patch.set_facecolor(c); patch.set_alpha(0.6)
    ax.set_ylabel('R²'); ax.set_title(f'{label} — R² by Formula', fontweight='bold')
    ax.tick_params(axis='x', rotation=15)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Score-Combination Formula Tuning (M2 & M3)', fontsize=14, fontweight='bold')
plt.tight_layout()
savefig('optuna_formula_search.png')
plt.show()

print('\nBest params found:')
print(f'  M2: formula={best_params_M2["formula"]}, w={best_params_M2["w"]:.4f}')
print(f'  M3: formula={best_params_M3["formula"]}, w={best_params_M3["w"]:.4f}')


### Apply tuned formulas to M2 / M3 OOF predictions

This overwrites `ALL_RESULTS['M2']` and `ALL_RESULTS['M3']` predictions with the
Optuna-tuned formula, so every downstream plot (confusion matrix, frieze, scatter,
comparison table) reflects the improved scoring — no retraining required.


In [ ]:
def rebuild_oof_with_formula(raw_ord, raw_reg, raw_idx, formula_name, w):
    """Re-assemble OOF arrays (aligned with y_raw order) using a chosen formula."""
    pred_raw = FORMULAS[formula_name](raw_ord, raw_reg, w)
    oof_pred_all = np.full(len(y_raw), np.nan, dtype=np.float32)
    oof_pred_all[raw_idx] = pred_raw
    oof_true_all = IDX_TO_SCORE[y_raw]
    return oof_pred_all, oof_true_all


for model_id, raw_ord, raw_reg, raw_idx, best_params in [
    ('M2', raw_ord_M2, raw_reg_M2, raw_idx_M2, best_params_M2),
    ('M3', raw_ord_M3, raw_reg_M3, raw_idx_M3, best_params_M3),
]:
    oof_pred_all, oof_true_all = rebuild_oof_with_formula(
        raw_ord, raw_reg, raw_idx, best_params['formula'], best_params['w'])

    new_metrics = compute_metrics(oof_pred_all, oof_true_all)
    old_metrics = ALL_RESULTS[model_id]['oof_metrics']

    print(f'{model_id}: R² {old_metrics["r2"]:.4f} → {new_metrics["r2"]:.4f}  '
          f'(formula={best_params["formula"]}, w={best_params["w"]:.3f})')

    # Recompute per-fold metrics under the new formula too, for consistency
    new_fold_metrics = []
    for fold_idx, (tr_idx, va_idx) in enumerate(fold_splits):
        fold_mask = np.isin(raw_idx, va_idx)
        fp = oof_pred_all[va_idx]
        ft = oof_true_all[va_idx]
        fm = compute_metrics(fp, ft)
        new_fold_metrics.append({**fm, 'fold': fold_idx+1,
                                 'best_epoch': ALL_RESULTS[model_id]['fold_metrics'][fold_idx]['best_epoch']})

    ALL_RESULTS[model_id]['oof_pred']     = oof_pred_all
    ALL_RESULTS[model_id]['oof_metrics']  = new_metrics
    ALL_RESULTS[model_id]['fold_metrics'] = new_fold_metrics
    ALL_RESULTS[model_id]['tuned_formula'] = best_params['formula']
    ALL_RESULTS[model_id]['tuned_w']       = best_params['w']

print('\n✔ ALL_RESULTS updated — all downstream cells (confusion matrix, frieze, etc.)')
print('  will now use the tuned scoring formula.')


## 12c · (Optional) Tuning Composite Loss Coefficients — Requires Retraining

Unlike the score-combination formula above, the composite loss weights
`(λ_ord, λ_huber, λ_rank)` affect **gradients during training**, so tuning them
means retraining M3 from scratch for each trial. This is expensive
(1 trial = 5-fold CV training), so we use a **reduced protocol**:
single train/val split (not full 5-fold CV) per trial, fewer max epochs,
then validate the winning config with full 5-fold CV.

**Skip this cell if M3's current performance is already satisfactory after the
formula tuning above — it is the most expensive cell in the notebook.**


In [ ]:
N_LOSS_TRIALS   = 20     # number of (λ_ord, λ_huber, λ_rank) combos to try
TRIAL_EPOCHS    = 60     # reduced epoch budget per trial (vs EPOCHS=300 for final run)
TRIAL_PATIENCE  = 12     # reduced ES patience per trial

# single stratified split for fast trial evaluation (not full 5-fold)
from sklearn.model_selection import train_test_split
trial_tr_idx, trial_va_idx = train_test_split(
    np.arange(len(y_raw)), test_size=0.2, stratify=y_raw, random_state=SEED)

print(f'Trial split: train={len(trial_tr_idx)}  val={len(trial_va_idx)}')


def loss_composite_weighted(ord_logits, reg_out, target, lam_ord, lam_huber, lam_rank):
    y_score = target.squeeze(1)
    l_ord   = loss_ordinal_bce(y_score, ord_logits)
    l_hub   = loss_huber(reg_out, target)
    l_rank  = loss_ranking(reg_out, target)
    return lam_ord * l_ord + lam_huber * l_hub + lam_rank * l_rank


def quick_train_trial(lam_ord, lam_huber, lam_rank):
    """Fast single-split training of M3 with given loss weights. Returns val R²."""
    lab_aug_f, feat_aug_f, y_aug_f = augment_fold(trial_tr_idx)
    lab_val_f  = X_lab[trial_va_idx]
    feat_val_f = feats_raw[trial_va_idx]
    y_val_f    = y_raw[trial_va_idx]

    tr_loader, va_loader = make_loaders(
        lab_aug_f, feat_aug_f, y_aug_f, lab_val_f, feat_val_f, y_val_f,
        batch_size=BATCH_SIZE)

    torch.manual_seed(SEED)
    model = M3_CompositeLoss().to(device)
    optimizer = optim.Adam(model.parameters(), lr=LR_INIT, weight_decay=1e-4)

    best_r2, wait = -np.inf, 0
    for epoch in range(TRIAL_EPOCHS):
        model.train()
        for img, feat, score, _ in tr_loader:
            img, feat, score = img.to(device), feat.to(device), score.to(device)
            optimizer.zero_grad()
            ord_l, reg_o = model(img, feat)
            loss = loss_composite_weighted(ord_l, reg_o, score, lam_ord, lam_huber, lam_rank)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

        val_preds, val_scores, _ = get_preds_and_targets(model, va_loader, 'M3', device)
        m = compute_metrics(val_preds, val_scores)
        if m['r2'] > best_r2:
            best_r2 = m['r2']; wait = 0
        else:
            wait += 1
            if wait >= TRIAL_PATIENCE:
                break
    return best_r2

print('Quick-trial function ready ✔')


In [ ]:
def loss_objective(trial):
    lam_ord   = trial.suggest_float('lam_ord',   0.1, 3.0)
    lam_huber = trial.suggest_float('lam_huber', 0.1, 3.0)
    lam_rank  = trial.suggest_float('lam_rank',  0.1, 3.0)
    r2 = quick_train_trial(lam_ord, lam_huber, lam_rank)
    return r2

study_loss = optuna.create_study(direction='maximize', sampler=TPESampler(seed=SEED))
study_loss.optimize(loss_objective, n_trials=N_LOSS_TRIALS, show_progress_bar=False)

best_loss_params = study_loss.best_params
print(f'\nBest composite loss weights found ({N_LOSS_TRIALS} trials):')
print(f'  λ_ord   = {best_loss_params["lam_ord"]:.3f}')
print(f'  λ_huber = {best_loss_params["lam_huber"]:.3f}')
print(f'  λ_rank  = {best_loss_params["lam_rank"]:.3f}')
print(f'  → trial R² = {study_loss.best_value:.4f}  '
      f'(vs current hardcoded 1.0/0.5/0.5)')

# Optimization history plot
fig, ax = plt.subplots(figsize=(9, 5))
trials_df = study_loss.trials_dataframe()
ax.plot(trials_df['number'], trials_df['value'], 'o-', alpha=0.6, color='steelblue')
ax.axhline(study_loss.best_value, color='crimson', ls='--', label=f'Best R²={study_loss.best_value:.4f}')
ax.set_xlabel('Trial'); ax.set_ylabel('Val R² (single split, reduced epochs)')
ax.set_title('Composite Loss Weight Search', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); savefig('optuna_loss_weights.png'); plt.show()


### Retrain M3 with tuned loss weights (full 5-fold CV)

Only run this if the trial search above found weights meaningfully better than the
defaults. This repeats the full M3 training protocol (5-fold, 300 epochs, patience=30)
with the new `(λ_ord, λ_huber, λ_rank)`, then **overwrites `ALL_RESULTS['M3']`** so all
downstream plots reflect the fully-tuned model (both loss weights AND score formula).


In [ ]:
RETRAIN_M3_WITH_TUNED_LOSS = True   # ← set True to actually retrain (expensive)

if RETRAIN_M3_WITH_TUNED_LOSS:
    lam_ord_final   = best_loss_params['lam_ord']
    lam_huber_final = best_loss_params['lam_huber']
    lam_rank_final  = best_loss_params['lam_rank']

    def loss_composite(ord_logits, reg_out, target):
        # Override global loss_composite with tuned weights for the retrain below
        y_score = target.squeeze(1)
        l_ord   = loss_ordinal_bce(y_score, ord_logits)
        l_hub   = loss_huber(reg_out, target)
        l_rank  = loss_ranking(reg_out, target)
        return lam_ord_final * l_ord + lam_huber_final * l_hub + lam_rank_final * l_rank

    print('Retraining M3 — Full 5-fold CV with tuned composite loss weights')
    print(f'  λ_ord={lam_ord_final:.3f}  λ_huber={lam_huber_final:.3f}  λ_rank={lam_rank_final:.3f}')

    oof_pred_all  = np.full(len(y_raw), np.nan, dtype=np.float32)
    oof_true_all  = IDX_TO_SCORE[y_raw]
    fold_hists, fold_metrics_list = [], []
    raw_ord_M3_retrained, raw_reg_M3_retrained, raw_idx_M3_retrained = [], [], []

    t0 = time.time()
    for fold_idx, (tr_idx, va_idx) in enumerate(fold_splits):
        print(f'\n  ── Fold {fold_idx+1}/{K_FOLDS} ────────────────────────────')
        oof_p, oof_t, oof_l, hist, best_ep = train_one_fold(
            M3_CompositeLoss, 'M3', fold_idx, tr_idx, va_idx)
        oof_pred_all[va_idx] = oof_p
        fold_hists.append(hist)
        m = compute_metrics(oof_p, oof_t)
        fold_metrics_list.append({**m, 'fold': fold_idx+1, 'best_epoch': best_ep+1})

    elapsed = (time.time() - t0) / 60
    oof_metrics = compute_metrics(oof_pred_all, oof_true_all)
    print(f'\n✔ M3 retrained in {elapsed:.1f} min  |  New OOF R²={oof_metrics["r2"]:.4f}')

    # Re-apply the best score formula (from section 12b) on top of the retrained heads
    raw_ord_M3_new, raw_reg_M3_new, raw_true_M3_new, raw_idx_M3_new = collect_raw_heads(M3_CompositeLoss, 'M3')
    oof_pred_final, oof_true_final = rebuild_oof_with_formula(
        raw_ord_M3_new, raw_reg_M3_new, raw_idx_M3_new,
        best_params_M3['formula'], best_params_M3['w'])
    final_metrics = compute_metrics(oof_pred_final, oof_true_final)

    print(f'  After re-applying tuned formula: R²={final_metrics["r2"]:.4f}')

    ALL_RESULTS['M3']['oof_pred']      = oof_pred_final
    ALL_RESULTS['M3']['oof_metrics']   = final_metrics
    ALL_RESULTS['M3']['fold_hist']     = fold_hists
    ALL_RESULTS['M3']['tuned_lam_ord']   = lam_ord_final
    ALL_RESULTS['M3']['tuned_lam_huber'] = lam_huber_final
    ALL_RESULTS['M3']['tuned_lam_rank']  = lam_rank_final
    print('  ✔ ALL_RESULTS[\'M3\'] fully updated (tuned loss + tuned formula)')
else:
    print('Skipped — set RETRAIN_M3_WITH_TUNED_LOSS=True to run full retraining.')


## 13 · Metric Comparison Bar Charts (OOF)

In [ ]:
COLORS = ['#4e79a7','#f28e2b','#59a14f','#e15759']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

metrics_to_plot = [
    ('r2',   'R² Score',         True),
    ('rmse', 'RMSE',             False),
    ('rho',  'Spearman ρ',       True),
    ('tau',  'Kendall τ',        True),
    ('acc0', 'Acc@0 (%)',        True),
    ('acc1', 'Acc@1 (%)',        True),
]

model_ids = list(MODEL_LABELS.keys())
xlabels   = [MODEL_LABELS[m] for m in model_ids]

for ax, (metric, title, higher_is_better) in zip(axes, metrics_to_plot):
    vals = []
    stds = []
    for mid in model_ids:
        fm = pd.DataFrame(ALL_RESULTS[mid]['fold_metrics'])
        scale = 100 if 'acc' in metric else 1
        vals.append(fm[metric].mean() * scale)
        stds.append(fm[metric].std() * scale)
    bars = ax.bar(range(len(model_ids)), vals, color=COLORS, edgecolor='black', width=0.6)
    ax.errorbar(range(len(model_ids)), vals, yerr=stds,
                fmt='none', color='black', capsize=5, capthick=2)
    ax.set_xticks(range(len(model_ids)))
    ax.set_xticklabels([m for m in model_ids], fontsize=11, fontweight='bold')
    ax.set_title(f'{title} (mean ± std across folds)', fontsize=12, fontweight='bold')
    ax.set_ylabel('% ' if 'acc' in metric else title)
    ax.grid(axis='y', alpha=0.3)
    # annotate
    for bar, v, s in zip(bars, vals, stds):
        ax.text(bar.get_x()+bar.get_width()/2, v+s+0.002,
                f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')
    arrow = '↑ better' if higher_is_better else '↓ better'
    ax.set_xlabel(arrow, fontsize=9, style='italic', color='gray')

plt.suptitle('Ablation Study — OOF Metric Comparison (5-Fold CV)',
             fontsize=16, fontweight='bold')
plt.tight_layout()
savefig('metric_comparison.png')
plt.show()


## 14 · Per-Fold R² Distribution (Variance / Stability)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, ylabel in [(axes[0],'r2','R²'), (axes[1],'rmse','RMSE')]:
    data = [pd.DataFrame(ALL_RESULTS[mid]['fold_metrics'])[metric].values
            for mid in model_ids]
    bp = ax.boxplot(data, patch_artist=True, notch=False,
                    medianprops=dict(color='black', linewidth=2.5))
    for patch, color in zip(bp['boxes'], COLORS):
        patch.set_facecolor(color); patch.set_alpha(0.7)
    ax.set_xticklabels(model_ids, fontsize=11, fontweight='bold')
    ax.set_ylabel(ylabel); ax.set_title(f'Per-fold {ylabel} — stability check',
                                         fontsize=12, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Per-Fold Metric Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
savefig('fold_stability.png')
plt.show()


## 15 · Training Curves — All Folds per Model

In [ ]:
fold_colors_tab = plt.cm.tab10(np.arange(K_FOLDS))

for model_id in model_ids:
    fhists = ALL_RESULTS[model_id]['fold_hist']
    fig, axes = plt.subplots(2, 3, figsize=(20, 10))
    axes = axes.flatten()

    metrics_curves = [
        ('train_loss', 'Train Loss', False),
        ('val_r2',     'Val R²',     False),
        ('val_rmse',   'Val RMSE',   False),
        ('val_rho',    'Val Spearman ρ', False),
        ('val_acc0',   'Val Acc@0 (%)', True),
        ('val_acc1',   'Val Acc@1 (%)', True),
    ]
    for ax, (key, title, pct) in zip(axes, metrics_curves):
        for fi, h in enumerate(fhists):
            vals = np.array(h[key]) * (100 if pct else 1)
            ax.plot(np.arange(1, len(vals)+1), vals,
                    lw=1.5, color=fold_colors_tab[fi], alpha=0.8, label=f'Fold {fi+1}')
        ax.set_title(title, fontsize=11, fontweight='bold')
        ax.set_xlabel('Epoch'); ax.grid(alpha=0.3)
        ax.legend(fontsize=8, ncol=2)

    plt.suptitle(f'{MODEL_LABELS[model_id]} — Training Curves (5 Folds)',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    savefig(f'curves_{model_id}.png')
    plt.show()


## 16 · Predicted vs True — All Models

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 13))
axes = axes.flatten()

for ax, model_id, color in zip(axes, model_ids, COLORS):
    res = ALL_RESULTS[model_id]
    pred = res['oof_pred']; true = res['oof_true']
    m = res['oof_metrics']
    jitter = np.random.normal(0, 0.06, size=true.shape)
    sc = ax.scatter(true+jitter, pred, c=np.abs(pred-true),
                    cmap='RdYlGn_r', vmin=0, vmax=2.5,
                    alpha=0.55, s=20, edgecolors='none')
    ax.plot([1,9],[1,9], 'k--', lw=1.5, label='Perfect')
    ax.fill_between([1,9],[0,8],[2,10], alpha=0.08, color='green', label='Within ±1')
    plt.colorbar(sc, ax=ax, label='|error|')
    ax.set_xlim(0.5,9.5); ax.set_ylim(0.5,9.5)
    ax.set_xlabel('True Score'); ax.set_ylabel('Predicted Score')
    ax.set_title(f'{MODEL_LABELS[model_id]}\nR²={m["r2"]:.3f}  ρ={m["rho"]:.3f}  '
                 f'RMSE={m["rmse"]:.3f}  Acc@1={m["acc1"]*100:.1f}%',
                 fontsize=11, fontweight='bold')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.suptitle('OOF: Predicted vs True Score — All Models',
             fontsize=15, fontweight='bold')
plt.tight_layout()
savefig('scatter_all_models.png')
plt.show()


## 17 · Confusion Matrices (Rounded Predictions)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(28, 12))
score_labels_str = [str(i) for i in range(1,10)]

for col, model_id in enumerate(model_ids):
    res   = ALL_RESULTS[model_id]
    pred  = np.clip(np.round(res['oof_pred']).astype(int), 1, 9)
    true  = res['oof_true'].astype(int)
    m     = res['oof_metrics']
    cm    = confusion_matrix(true, pred, labels=list(range(1,10)))
    cm_n  = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-9)

    # Raw counts
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=score_labels_str, yticklabels=score_labels_str,
                cbar=False, ax=axes[0,col], linewidths=0.3)
    axes[0,col].set_xlabel('Predicted'); axes[0,col].set_ylabel('True')
    axes[0,col].set_title(f'{model_id} — Counts\nAcc@1={m["acc1"]*100:.1f}%',
                           fontsize=10, fontweight='bold')

    # Row-normalised
    sns.heatmap(cm_n, annot=True, fmt='.2f', cmap='Greens', vmin=0, vmax=1,
                xticklabels=score_labels_str, yticklabels=score_labels_str,
                cbar=False, ax=axes[1,col], linewidths=0.3)
    axes[1,col].set_xlabel('Predicted'); axes[1,col].set_ylabel('True')
    axes[1,col].set_title(f'{model_id} — Row-Normalised (Recall)',
                           fontsize=10, fontweight='bold')

plt.suptitle('Confusion Matrices — OOF Rounded Predictions vs True Scores',
             fontsize=15, fontweight='bold')
plt.tight_layout()
savefig('confusion_matrices_all.png')
plt.show()


## 18 · Per-Score Prediction Distribution (Boxplots)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()
scores_range = list(range(1, 10))

for ax, model_id, color in zip(axes, model_ids, COLORS):
    res   = ALL_RESULTS[model_id]
    pred  = res['oof_pred']
    true  = res['oof_true']
    data  = [pred[true == s] for s in scores_range]
    bp    = ax.boxplot(data, positions=scores_range, widths=0.6, patch_artist=True,
                       boxprops=dict(facecolor=color, alpha=0.6, edgecolor='black'),
                       medianprops=dict(color='black', linewidth=2),
                       whiskerprops=dict(color='black'), capprops=dict(color='black'))
    for s in scores_range:
        ax.axhline(y=s, color='green', linestyle='--', alpha=0.4, lw=1)
    ax.set_xlabel('True Score'); ax.set_ylabel('Predicted Score')
    m = res['oof_metrics']
    ax.set_title(f'{MODEL_LABELS[model_id]}\nR²={m["r2"]:.3f}  RMSE={m["rmse"]:.3f}',
                 fontsize=11, fontweight='bold')
    ax.set_xticks(scores_range)
    ax.set_ylim(0, 10); ax.grid(axis='y', alpha=0.3)

plt.suptitle('OOF Prediction Distribution per True Score Class',
             fontsize=15, fontweight='bold')
plt.tight_layout()
savefig('boxplots_per_score.png')
plt.show()


## 19 · Prediction Friezes — One Row per Score, Max 15 Images per Row

Each row = one true score class. Images sorted by predicted value.  
Border color: 🟢 error ≤ 0.5 | 🟠 error ≤ 1.0 | 🔴 error > 1.0


In [ ]:
def draw_frieze(model_id, max_per_row=MAX_PER_ROW):
    """
    Draw one figure per model.
    Layout: one row per true score class (1-9).
    Each row has a label column + up to max_per_row image columns.
    Images within a row are sorted by predicted value (ascending).
    Rows with fewer than max_per_row images use a GridSpec sized to that
    row's actual column count (label + n_show), so we never index out of bounds.
    Border color: green if |err|<=0.5, orange if |err|<=1.0, red otherwise.
    """
    res    = ALL_RESULTS[model_id]
    pred   = res['oof_pred']
    true   = res['oof_true']
    scores = sorted(np.unique(true.astype(int)).tolist())
    n_rows = len(scores)
    n_cols_fig = max_per_row + 1   # label + images, used only for figure width

    fig = plt.figure(figsize=(n_cols_fig * 1.10, n_rows * 1.70))
    outer_gs = gridspec.GridSpec(n_rows, 1, figure=fig, hspace=0.55)

    for row_i, score in enumerate(scores):
        mask      = true.astype(int) == score
        p_row     = pred[mask]
        orig_idx  = np.where(mask)[0]

        # sort by predicted value
        order      = np.argsort(p_row)
        p_sorted   = p_row[order]
        idx_sorted = orig_idx[order]

        n_show   = min(len(idx_sorted), max_per_row)
        p_show   = p_sorted[:n_show]
        idx_show = idx_sorted[:n_show]

        # GridSpec sized exactly to this row's content: label + n_show images.
        # (Previously sized to max_per_row+1 always, causing out-of-bounds
        #  when n_show < max_per_row.)
        inner = gridspec.GridSpecFromSubplotSpec(
            1, n_show + 1,
            subplot_spec=outer_gs[row_i],
            wspace=0.04)

        # ── Score label ──────────────────────────────────────────────────────
        ax_lbl = fig.add_subplot(inner[0])
        ax_lbl.axis('off')
        clr = plt.cm.RdYlGn(row_i / max(n_rows - 1, 1))
        ax_lbl.text(0.5, 0.5,
                    f'Score {score}\n(n={int(mask.sum())})',
                    ha='center', va='center',
                    fontsize=10, fontweight='bold', color=clr,
                    transform=ax_lbl.transAxes)

        # ── Images ───────────────────────────────────────────────────────────
        for col_i in range(n_show):
            orig = idx_show[col_i]
            p_val = p_show[col_i]
            err   = abs(p_val - score)
            border = 'limegreen' if err <= 0.5 else ('orange' if err <= 1.0 else 'red')

            ax = fig.add_subplot(inner[col_i + 1])
            ax.imshow(lab_to_rgb(X_lab[orig]))
            ax.set_title(f'{p_val:.1f}', fontsize=7, color=border, pad=1.5, fontweight='bold')
            ax.set_xticks([]); ax.set_yticks([])
            for sp in ax.spines.values():
                sp.set_edgecolor(border); sp.set_linewidth(1.8)

        # No more "fill unused columns" loop — the GridSpec is already
        # sized exactly to n_show, so there are no unused columns to fill.

    fig.suptitle(
        f'{MODEL_LABELS[model_id]} — Prediction Frieze  '
        f'(max {max_per_row} images / row, sorted by predicted score)\n'
        '🟢 err≤0.5 | 🟠 err≤1.0 | 🔴 err>1.0',
        fontsize=12, fontweight='bold', y=1.01)

    out_path = f'training/figures/frieze_{model_id}.png'
    plt.savefig(out_path, dpi=120, bbox_inches='tight')
    print(f'  💾 {out_path}')
    plt.show()
    plt.close(fig)


for mid in model_ids:
    print(f'\nDrawing frieze for {MODEL_LABELS[mid]} ...')
    draw_frieze(mid, max_per_row=MAX_PER_ROW)


## 20 · Per-Class Statistics Summary — All Models

In [ ]:
print('\n' + '='*80)
print('  PER-CLASS STATISTICS SUMMARY (OOF)')
print('='*80)

for model_id in model_ids:
    res  = ALL_RESULTS[model_id]
    pred = res['oof_pred']; true = res['oof_true']
    print(f'\n  {MODEL_LABELS[model_id]}')
    print(f'  {"─"*60}')
    rows_cls = []
    for s in range(1, 10):
        mask = true.astype(int) == s
        if not mask.any(): continue
        p = pred[mask]; t = true[mask]
        rows_cls.append({
            'Score': s,
            'N':     int(mask.sum()),
            'Mean pred ± std': f'{p.mean():.3f} ± {p.std():.3f}',
            'MAE':   f'{np.mean(np.abs(p-t)):.3f}',
            'Acc@1': f'{within_acc(p,t,1)*100:.0f}%',
        })
    print(pd.DataFrame(rows_cls).to_string(index=False))


## 21 · Final Summary & Key Takeaways

In [ ]:
print('\n' + '='*75)
print('  FINAL OOF RESULTS — ABLATION STUDY'.center(75))
print('='*75)
print(f'{"Model":<25} {"R²":>7} {"RMSE":>7} {"ρ":>7} {"τ":>7} {"Acc@0":>8} {"Acc@1":>8}')
print('-'*75)
for mid in model_ids:
    m = ALL_RESULTS[mid]['oof_metrics']
    print(f'{MODEL_LABELS[mid]:<25} '
          f'{m["r2"]:>7.4f} {m["rmse"]:>7.4f} '
          f'{m["rho"]:>7.4f} {m["tau"]:>7.4f} '
          f'{m["acc0"]*100:>7.1f}% {m["acc1"]*100:>7.1f}%')
print('='*75)

# Δ gain over baseline
m0 = ALL_RESULTS['M0']['oof_metrics']
print('\n  Δ improvement over M0 (Baseline):')
for mid in ['M1','M2','M3']:
    m = ALL_RESULTS[mid]['oof_metrics']
    dr2   = m['r2']   - m0['r2']
    drmse = m['rmse'] - m0['rmse']
    drho  = m['rho']  - m0['rho']
    da1   = m['acc1'] - m0['acc1']
    sign_r2   = '+' if dr2   >= 0 else ''
    sign_rmse = '+' if drmse >= 0 else ''
    sign_rho  = '+' if drho  >= 0 else ''
    sign_a1   = '+' if da1   >= 0 else ''
    print(f'  {MODEL_LABELS[mid]:<25} '
          f'ΔR²={sign_r2}{dr2:+.4f}  '
          f'ΔRMSE={sign_rmse}{drmse:+.4f}  '
          f'Δρ={sign_rho}{drho:+.4f}  '
          f'ΔAcc@1={sign_a1}{da1*100:+.1f}%')
print('='*75)


In [ ]:
# Dependency check
_required = [
    'nn', 'torch', 'optim', 'optuna', 'TPESampler', 'np', 'pd', 'time', 'plt',
    'device', 'BATCH_SIZE', 'EPOCHS', 'K_FOLDS', 'PATIENCE_ES', 'PATIENCE_LR',
    'LR_INIT', 'LR_MIN', 'LR_FACTOR', 'SEED', 'IDX_TO_SCORE',
    'CNNBackbone', 'CNN_DIM', 'NUM_ORD',
    'X_lab', 'feats_raw', 'y_raw', 'fold_splits', 'augment_fold',
    'ZucchiniDataset', 'make_loaders', 'get_preds_and_targets', 'train_one_fold',
    'loss_ordinal_bce', 'loss_huber', 'loss_ranking', 'compute_metrics',
    'ALL_RESULTS', 'MODEL_LABELS',
]
_missing = [n for n in _required if n not in globals()]
if _missing:
    raise NameError(
        'Missing from kernel: ' + ', '.join(_missing) +
        '. Run the base ablation-study notebook cells first (through section 11 at '
        'least, and 12b/12c for the Step 3-5 helpers below), in this same kernel.'
    )
print('All base dependencies found in kernel.')

_optional = ['FORMULAS', 'collect_raw_heads', 'run_optuna_search',
             'rebuild_oof_with_formula', 'loss_composite_weighted']
_missing_opt = [n for n in _optional if n not in globals()]
if _missing_opt:
    print('Missing (needed later, for ensemble-formula tuning): ' + ', '.join(_missing_opt))
    print('Run section 12b (formula tuning) and 12c (loss_composite_weighted def) cells first.')
else:
    print('Formula-tuning helpers also found -- full pipeline (Steps 1-5) can run.')


In [ ]:
class M4_NoColor(nn.Module):
    """M3's architecture minus the 54-dim color-feature branch.
    CNN backbone -> shared_fc(256->128) -> (ord_head, reg_head). No feat fusion."""
    def __init__(self):
        super().__init__()
        self.backbone = CNNBackbone()
        self.shared_fc = nn.Sequential(
            nn.Linear(CNN_DIM, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.35),
        )
        self.ord_head = nn.Linear(128, NUM_ORD)   # ordinal logits
        self.reg_head = nn.Linear(128, 1)         # regression output

    def forward(self, img, feat=None):
        # feat accepted-but-unused: keeps this class compatible with the shared
        # training engine, which always feeds (img, feat) pairs from the dataset.
        cnn_out = self.backbone(img)
        fused   = self.shared_fc(cnn_out)
        return self.ord_head(fused), self.reg_head(fused)   # (B, NUM_ORD), (B, 1)

    @staticmethod
    def predict_score(ord_logits, reg_out):
        """Placeholder 0.7/0.3 blend, used only to pick the 'best epoch' checkpoint
        during CV training. The real formula + weight are tuned post-hoc in Step 3."""
        ord_score = 1.0 + torch.sigmoid(ord_logits).sum(dim=1, keepdim=True)
        reg_score = reg_out.clamp(1.0, 9.0)
        return 0.7 * ord_score + 0.3 * reg_score


_n_params_m4 = sum(p.numel() for p in M4_NoColor().parameters() if p.requires_grad)
print(f'M4 -- No Color Feats : {_n_params_m4:,} trainable params')


In [ ]:
N_LOSS_TRIALS_M4   = 20      # combos of (lambda_ord, lambda_huber, lambda_rank) to try
TRIAL_EPOCHS_M4    = 60      # reduced epoch budget per trial (vs EPOCHS for final run)
TRIAL_PATIENCE_M4  = 12      # reduced ES patience per trial

from sklearn.model_selection import train_test_split
trial_tr_idx_M4, trial_va_idx_M4 = train_test_split(
    np.arange(len(y_raw)), test_size=0.2, stratify=y_raw, random_state=SEED)

print(f'Trial split: train={len(trial_tr_idx_M4)}  val={len(trial_va_idx_M4)}')

# Redefine standalone in case section 12c wasn't run in this kernel
def loss_composite_weighted(ord_logits, reg_out, target, lam_ord, lam_huber, lam_rank):
    y_score = target.squeeze(1)
    l_ord   = loss_ordinal_bce(y_score, ord_logits)
    l_hub   = loss_huber(reg_out, target)
    l_rank  = loss_ranking(reg_out, target)
    return lam_ord * l_ord + lam_huber * l_hub + lam_rank * l_rank


def quick_train_trial_M4(lam_ord, lam_huber, lam_rank):
    """Fast single-split training of M4 with given loss weights. Returns best val R2."""
    lab_aug_f, feat_aug_f, y_aug_f = augment_fold(trial_tr_idx_M4)
    lab_val_f  = X_lab[trial_va_idx_M4]
    feat_val_f = feats_raw[trial_va_idx_M4]
    y_val_f    = y_raw[trial_va_idx_M4]

    tr_loader, va_loader = make_loaders(
        lab_aug_f, feat_aug_f, y_aug_f, lab_val_f, feat_val_f, y_val_f,
        batch_size=BATCH_SIZE)

    torch.manual_seed(SEED)
    model = M4_NoColor().to(device)
    optimizer = optim.Adam(model.parameters(), lr=LR_INIT, weight_decay=1e-4)

    best_r2, wait = -np.inf, 0
    for epoch in range(TRIAL_EPOCHS_M4):
        model.train()
        for img, feat, score, _ in tr_loader:
            img, feat, score = img.to(device), feat.to(device), score.to(device)
            optimizer.zero_grad()
            ord_l, reg_o = model(img, feat)
            loss = loss_composite_weighted(ord_l, reg_o, score, lam_ord, lam_huber, lam_rank)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

        val_preds, val_scores, _ = get_preds_and_targets(model, va_loader, 'M4', device)
        m = compute_metrics(val_preds, val_scores)
        if m['r2'] > best_r2:
            best_r2 = m['r2']; wait = 0
        else:
            wait += 1
            if wait >= TRIAL_PATIENCE_M4:
                break
    return best_r2

print('Quick-trial function ready.')


In [ ]:
def loss_objective_M4(trial):
    lam_ord   = trial.suggest_float('lam_ord',   0.1, 3.0)
    lam_huber = trial.suggest_float('lam_huber', 0.1, 3.0)
    lam_rank  = trial.suggest_float('lam_rank',  0.1, 3.0)
    return quick_train_trial_M4(lam_ord, lam_huber, lam_rank)

study_loss_M4 = optuna.create_study(direction='maximize', sampler=TPESampler(seed=SEED))
study_loss_M4.optimize(loss_objective_M4, n_trials=N_LOSS_TRIALS_M4, show_progress_bar=False)

best_loss_params_M4 = study_loss_M4.best_params
print(f'Best composite loss weights for M4 ({N_LOSS_TRIALS_M4} trials):')
print(f'  lambda_ord   = {best_loss_params_M4["lam_ord"]:.3f}')
print(f'  lambda_huber = {best_loss_params_M4["lam_huber"]:.3f}')
print(f'  lambda_rank  = {best_loss_params_M4["lam_rank"]:.3f}')
print(f'  -> trial R2 = {study_loss_M4.best_value:.4f}  (vs default hardcoded 1.0/0.5/0.5)')

fig, ax = plt.subplots(figsize=(9, 5))
trials_df = study_loss_M4.trials_dataframe()
ax.plot(trials_df['number'], trials_df['value'], 'o-', alpha=0.6, color='steelblue')
ax.axhline(study_loss_M4.best_value, color='crimson', ls='--',
           label=f'Best R2={study_loss_M4.best_value:.4f}')
ax.set_xlabel('Trial'); ax.set_ylabel('Val R2 (single split, reduced epochs)')
ax.set_title('M4 -- Composite Loss Weight Search', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
lam_ord_M4   = best_loss_params_M4['lam_ord']
lam_huber_M4 = best_loss_params_M4['lam_huber']
lam_rank_M4  = best_loss_params_M4['lam_rank']

def loss_composite(ord_logits, reg_out, target):
    """Global loss_composite rebound to M4's tuned weights (see markdown above)."""
    y_score = target.squeeze(1)
    l_ord   = loss_ordinal_bce(y_score, ord_logits)
    l_hub   = loss_huber(reg_out, target)
    l_rank  = loss_ranking(reg_out, target)
    return lam_ord_M4 * l_ord + lam_huber_M4 * l_hub + lam_rank_M4 * l_rank

print('Retraining M4 -- Full 5-fold CV with tuned composite loss weights')
print(f'  lambda_ord={lam_ord_M4:.3f}  lambda_huber={lam_huber_M4:.3f}  lambda_rank={lam_rank_M4:.3f}')

oof_pred_all_M4  = np.full(len(y_raw), np.nan, dtype=np.float32)
oof_true_all_M4  = IDX_TO_SCORE[y_raw]
fold_hists_M4, fold_metrics_list_M4 = [], []

t0 = time.time()
for fold_idx, (tr_idx, va_idx) in enumerate(fold_splits):
    print(f'Fold {fold_idx+1}/{K_FOLDS}')
    oof_p, oof_t, oof_l, hist, best_ep = train_one_fold(
        M4_NoColor, 'M4', fold_idx, tr_idx, va_idx)
    oof_pred_all_M4[va_idx] = oof_p
    fold_hists_M4.append(hist)
    m = compute_metrics(oof_p, oof_t)
    fold_metrics_list_M4.append({**m, 'fold': fold_idx+1, 'best_epoch': best_ep+1})

elapsed = (time.time() - t0) / 60
oof_metrics_M4 = compute_metrics(oof_pred_all_M4, oof_true_all_M4)
print(f'M4 trained in {elapsed:.1f} min')
print(f'  OOF R2={oof_metrics_M4["r2"]:.4f}  RMSE={oof_metrics_M4["rmse"]:.4f}  '
      f'rho={oof_metrics_M4["rho"]:.4f}  Acc@0={oof_metrics_M4["acc0"]*100:.1f}%  '
      f'Acc@1={oof_metrics_M4["acc1"]*100:.1f}%')

ALL_RESULTS['M4'] = {
    'oof_pred':     oof_pred_all_M4,
    'oof_true':     oof_true_all_M4,
    'fold_hist':    fold_hists_M4,
    'fold_metrics': fold_metrics_list_M4,
    'oof_metrics':  oof_metrics_M4,
    'tuned_lam_ord':   lam_ord_M4,
    'tuned_lam_huber': lam_huber_M4,
    'tuned_lam_rank':  lam_rank_M4,
}


In [ ]:
raw_ord_M4, raw_reg_M4, raw_true_M4, raw_idx_M4 = collect_raw_heads(M4_NoColor, 'M4')
print(f'{raw_ord_M4.shape[0]} samples, ord_logits shape={raw_ord_M4.shape}')

study_M4, best_params_M4, tuned_pred_M4, tuned_metrics_M4, base_metrics_M4 = \
    run_optuna_search(raw_ord_M4, raw_reg_M4, raw_true_M4, 'M4 -- No Color Feats', n_trials=150)


In [ ]:
oof_pred_all_M4b, oof_true_all_M4b = rebuild_oof_with_formula(
    raw_ord_M4, raw_reg_M4, raw_idx_M4, best_params_M4['formula'], best_params_M4['w'])

new_metrics_M4 = compute_metrics(oof_pred_all_M4b, oof_true_all_M4b)
old_metrics_M4 = ALL_RESULTS['M4']['oof_metrics']
print(f'M4: R2 {old_metrics_M4["r2"]:.4f} -> {new_metrics_M4["r2"]:.4f}  '
      f'(formula={best_params_M4["formula"]}, w={best_params_M4["w"]:.3f})')

new_fold_metrics_M4 = []
for fold_idx, (tr_idx, va_idx) in enumerate(fold_splits):
    fp = oof_pred_all_M4b[va_idx]
    ft = oof_true_all_M4b[va_idx]
    fm = compute_metrics(fp, ft)
    new_fold_metrics_M4.append({**fm, 'fold': fold_idx+1,
                                 'best_epoch': ALL_RESULTS['M4']['fold_metrics'][fold_idx]['best_epoch']})

ALL_RESULTS['M4']['oof_pred']      = oof_pred_all_M4b
ALL_RESULTS['M4']['oof_metrics']   = new_metrics_M4
ALL_RESULTS['M4']['fold_metrics']  = new_fold_metrics_M4
ALL_RESULTS['M4']['tuned_formula'] = best_params_M4['formula']
ALL_RESULTS['M4']['tuned_w']       = best_params_M4['w']

print('ALL_RESULTS["M4"] fully updated -- tuned composite loss + tuned ensemble formula.')


In [ ]:
MODEL_LABELS_EXT = dict(MODEL_LABELS)
MODEL_LABELS_EXT['M4'] = 'M4 -- No Color, Tuned Loss+Formula'

rows = []
for mid, label in MODEL_LABELS_EXT.items():
    m = ALL_RESULTS[mid]['oof_metrics']
    rows.append({
        'Model':       label,
        'R2 (OOF)':    f"{m['r2']:.4f}",
        'RMSE (OOF)':  f"{m['rmse']:.4f}",
        'Spearman rho':f"{m['rho']:.4f}",
        'Kendall tau': f"{m['tau']:.4f}",
        'Acc@0 (%)':   f"{m['acc0']*100:.1f}",
        'Acc@1 (%)':   f"{m['acc1']*100:.1f}",
    })

df_cmp_ext = pd.DataFrame(rows)
print('COMPARISON TABLE -- M0 -> M4  (M4 = no color feats, tuned composite loss + ensemble formula)')
print(df_cmp_ext.to_string(index=False))

m0 = ALL_RESULTS['M0']['oof_metrics']
print()
print('Delta improvement over M0 (Baseline):')
for mid in ['M1', 'M2', 'M3', 'M4']:
    if mid not in ALL_RESULTS:
        continue
    m     = ALL_RESULTS[mid]['oof_metrics']
    dr2   = m['r2']   - m0['r2']
    drmse = m['rmse'] - m0['rmse']
    drho  = m['rho']  - m0['rho']
    da1   = m['acc1'] - m0['acc1']
    print(f'  {MODEL_LABELS_EXT[mid]:<38} dR2={dr2:+.4f}  dRMSE={drmse:+.4f}  '
          f'drho={drho:+.4f}  dAcc@1={da1*100:+.1f}%')

import os
os.makedirs('training', exist_ok=True)
df_cmp_ext.to_csv('training/comparison_table_with_M4.csv', index=False)
print()
print('Saved -> training/comparison_table_with_M4.csv')


## 22 · Completing the Compositional Ablation Grid

Because this is an **incremental** ablation (each model builds on the previous
one), the composition mapping is not what the `M0`/`M1`/`M2`/`M3` names alone
suggest:

| Model | What it actually is | Composition |
|---|---|---|
| `M0` | CNN backbone, single head, plain MSE | baseline |
| `M1` | `M0` + 54-dim color features | **M1** |
| `M2` | `M1` + dual (ordinal+regression) head, `loss_dual_mse` | **M1+M2** |
| `M3` | `M2` + composite loss (`loss_composite`) | **M1+M2+M3** |
| `M4` *(section above, if run)* | Dual head + composite loss, **no** color features, but retrained with **Optuna‑tuned** loss weights + ensemble formula | **M2+M3**, but *tuned* — not a clean ablation isolate |

So `M1+M2` and `M1+M2+M3` already exist — they're just labelled `M2` and `M3`.
The one composition that's genuinely missing for a **fair, apples-to-apples**
comparison is **`M2+M3` trained with the exact same recipe as `M2`/`M3`**
(vanilla composite-loss weights, no Optuna tuning, no color branch). That's
what this section adds, then rolls everything up into one table with CV
(mean±std across the 5 folds) *and* OOF metrics for every metric — not just R².

This section only depends on cells through **§11/§12** (`ALL_RESULTS`,
`train_one_fold`, `compute_metrics`, `fold_splits`, etc.) — it does **not**
require the Optuna tuning sections (12b/12c) or the `M4` section to have been
run.

In [ ]:
# ── Dependency check for this section ────────────────────────────────────────
_required_23 = [
    'nn', 'torch', 'optim', 'np', 'pd', 'time', 'os',
    'device', 'BATCH_SIZE', 'EPOCHS', 'K_FOLDS', 'SEED', 'IDX_TO_SCORE',
    'CNNBackbone', 'CNN_DIM', 'NUM_ORD',
    'y_raw', 'fold_splits', 'augment_fold',
    'loss_ordinal_bce', 'loss_huber', 'loss_ranking', 'compute_metrics',
    'train_one_fold', 'get_preds_and_targets', 'ALL_RESULTS', 'MODEL_LABELS',
]
_missing_23 = [n for n in _required_23 if n not in globals()]
if _missing_23:
    raise NameError(
        'Missing from kernel: ' + ', '.join(_missing_23) +
        '. Run the base ablation-study cells first (through section 11/12, '
        'M0-M3) in this same kernel before running this section.'
    )
for _mid in ('M0', 'M1', 'M2', 'M3'):
    if _mid not in ALL_RESULTS:
        raise NameError(
            f"ALL_RESULTS['{_mid}']"
            " is missing -- run section 11 (Run All Four Models) first."
        )
print('All dependencies for the compositional-grid section found ✔')

### 22a · `M2+M3` architecture (Dual Head + Composite Loss, no color features)

In [ ]:
class M23_DualHead_CompositeLoss_NoColor(nn.Module):
    """The M2+M3 composition in isolation: same dual (ordinal+regression) head
    and composite loss as M3, but *without* the M1 color-feature fusion branch.
    Architecturally this is identical to M3 minus `feat_mlp` -- i.e. CNN
    backbone -> shared_fc(256->128) -> (ord_head, reg_head).

    Defined independently of the `M4_NoColor` class used in the Optuna-tuning
    section above so this cell can run standalone, and so it's never
    accidentally trained with tuned loss weights or a tuned ensemble formula.
    """
    def __init__(self):
        super().__init__()
        self.backbone = CNNBackbone()
        self.shared_fc = nn.Sequential(
            nn.Linear(CNN_DIM, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.35),
        )
        self.ord_head = nn.Linear(128, NUM_ORD)
        self.reg_head = nn.Linear(128, 1)

    def forward(self, img, feat=None):
        # feat accepted-but-unused: keeps this class compatible with the shared
        # training engine, which always feeds (img, feat) pairs from the dataset.
        cnn_out = self.backbone(img)
        fused   = self.shared_fc(cnn_out)
        return self.ord_head(fused), self.reg_head(fused)   # (B, NUM_ORD), (B, 1)

    @staticmethod
    def predict_score(ord_logits, reg_out):
        """Same 0.7×ordinal + 0.3×regression blend as M2/M3 -- kept identical
        so the comparison against M2/M3 isn't confounded by a different
        ensemble formula."""
        ord_score = 1.0 + torch.sigmoid(ord_logits).sum(dim=1, keepdim=True)
        reg_score = reg_out.clamp(1.0, 9.0)
        return 0.7 * ord_score + 0.3 * reg_score


def loss_composite_vanilla(ord_logits, reg_out, target):
    """The *original* M3 composite-loss weights (1.0·ord + 0.5·huber + 0.5·rank),
    pinned explicitly so this model is never affected by whatever
    Optuna-tuned weights the 12c/M4 sections may have rebound the global
    `loss_composite` name to."""
    y_score = target.squeeze(1)
    l_ord   = loss_ordinal_bce(y_score, ord_logits)
    l_hub   = loss_huber(reg_out, target)
    l_rank  = loss_ranking(reg_out, target)
    return 1.0 * l_ord + 0.5 * l_hub + 0.5 * l_rank


_n_params_23 = sum(p.numel() for p in M23_DualHead_CompositeLoss_NoColor().parameters()
                    if p.requires_grad)
print(f'M2+M3 (pure) -- Dual Head + Composite Loss, No Color : {_n_params_23:,} trainable params')

### 22b · Train `M2+M3` (pure) — 5-Fold CV, same recipe as M2/M3

In [ ]:
print('\n' + '='*70)
print('  M2+M3 -- Dual Head + Composite Loss (No Color Feats)'.center(70))
print('='*70)
print('  vanilla loss weights: lambda_ord=1.0  lambda_huber=0.5  lambda_rank=0.5')
print('  ensemble blend: 0.7 x ordinal + 0.3 x regression (same as M2/M3)')

oof_pred_all_23  = np.full(len(y_raw), np.nan, dtype=np.float32)
oof_true_all_23  = IDX_TO_SCORE[y_raw]
oof_label_all_23 = y_raw.copy()
fold_hists_23, fold_metrics_list_23 = [], []

t0 = time.time()
for fold_idx, (tr_idx, va_idx) in enumerate(fold_splits):
    print(f'\n  -- Fold {fold_idx+1}/{K_FOLDS} ----------------------------')
    oof_p, oof_t, oof_l, hist, best_ep = train_one_fold(
        M23_DualHead_CompositeLoss_NoColor, 'M23', fold_idx, tr_idx, va_idx,
        loss_fn=loss_composite_vanilla)

    oof_pred_all_23[va_idx] = oof_p
    fold_hists_23.append(hist)
    m = compute_metrics(oof_p, oof_t)
    fold_metrics_list_23.append({**m, 'fold': fold_idx+1, 'best_epoch': best_ep+1})

elapsed = (time.time() - t0) / 60
oof_metrics_23 = compute_metrics(oof_pred_all_23, oof_true_all_23)
print(f'\n  ✔ M2+M3 (pure) done in {elapsed:.1f} min')
print(f'  OOF R²={oof_metrics_23["r2"]:.4f}  RMSE={oof_metrics_23["rmse"]:.4f}  '
      f'ρ={oof_metrics_23["rho"]:.4f}  Acc@0={oof_metrics_23["acc0"]*100:.1f}%  '
      f'Acc@1={oof_metrics_23["acc1"]*100:.1f}%')

ALL_RESULTS['M23'] = {
    'oof_pred':     oof_pred_all_23,
    'oof_true':     oof_true_all_23,
    'oof_label':    oof_label_all_23,
    'fold_hist':    fold_hists_23,
    'fold_metrics': fold_metrics_list_23,
    'oof_metrics':  oof_metrics_23,
}
print("Stored -> ALL_RESULTS['M23']")

### 22c · Unified comparison — CV (mean±std, all metrics) + OOF, all compositions

In [ ]:
os.makedirs('training', exist_ok=True)

COMPOSITION_LABELS = {
    'M0':  'M0 -- Baseline (no color, single head, MSE)',
    'M1':  'M1 -- +Color Feats',
    'M2':  'M1+M2 -- +Color Feats +Dual Head',
    'M3':  'M1+M2+M3 -- +Color Feats +Dual Head +Composite Loss',
    'M23': 'M2+M3 -- Dual Head +Composite Loss (No Color)',
}

METRIC_COLS  = ['r2', 'rmse', 'rho', 'tau', 'acc0', 'acc1']
METRIC_NAMES = {'r2': 'R2', 'rmse': 'RMSE', 'rho': 'Spearman rho',
                'tau': 'Kendall tau', 'acc0': 'Acc@0 (%)', 'acc1': 'Acc@1 (%)'}
PCT_METRICS  = {'acc0', 'acc1'}

rows = []
for mid, label in COMPOSITION_LABELS.items():
    if mid not in ALL_RESULTS:
        print(f'  [skip] {label} -- ALL_RESULTS["{mid}"] not found (run its training cell first)')
        continue
    res = ALL_RESULTS[mid]
    m   = res['oof_metrics']
    fm  = pd.DataFrame(res['fold_metrics'])
    row = {'Composition': label}
    for mc in METRIC_COLS:
        scale  = 100 if mc in PCT_METRICS else 1
        fmt    = '.1f' if mc in PCT_METRICS else '.4f'
        oof_v  = m[mc] * scale
        cv_m   = fm[mc].mean() * scale
        cv_s   = fm[mc].std()  * scale
        row[f'{METRIC_NAMES[mc]} (OOF)']          = f'{oof_v:{fmt}}'
        row[f'{METRIC_NAMES[mc]} (CV mean+-std)'] = f'{cv_m:{fmt}}+-{cv_s:{fmt}}'
    rows.append(row)

df_compositions = pd.DataFrame(rows)

print('\n' + '='*130)
print('  ABLATION COMPOSITIONS -- OOF + 5-Fold CV Metrics (mean +/- std)'.center(130))
print('='*130)
print(df_compositions.to_string(index=False))
print('='*130)

df_compositions.to_csv('training/comparison_table_compositions.csv', index=False)
print('\nSaved -> training/comparison_table_compositions.csv')

# ── Delta vs baseline (R^2 and Acc@1), for quick reading ────────────────────
if 'M0' in ALL_RESULTS:
    m0 = ALL_RESULTS['M0']['oof_metrics']
    print('\nDelta vs M0 (Baseline), OOF:')
    for mid, label in COMPOSITION_LABELS.items():
        if mid == 'M0' or mid not in ALL_RESULTS:
            continue
        m = ALL_RESULTS[mid]['oof_metrics']
        dr2  = m['r2']   - m0['r2']
        da1  = m['acc1'] - m0['acc1']
        print(f'  {label:<55} dR2={dr2:+.4f}   dAcc@1={da1*100:+.1f}%')

## 23 · Full Single-Factor + Combination Grid (7 arms)

You asked for the baseline plus **every** one of these arms benchmarked:

| # | Arm | Color feats | Dual head | Composite loss | Status |
|---|---|---|---|---|---|
| 1 | Baseline | ✗ | ✗ | ✗ | `M0` — already exists |
| 2 | + Color feats | ✓ | ✗ | ✗ | `M1` — already exists |
| 3 | **+ Dual head** | ✗ | ✓ | ✗ | **NEW → `M2pure`** |
| 4 | **+ Composite loss** | ✗ | ✗ | ✓ | **NEW → `M3pure`** |
| 5 | + Color feats + Dual head + Composite loss | ✓ | ✓ | ✓ | `M3` (= M1+M2+M3) — already exists |
| 6 | + Color feats + Dual head | ✓ | ✓ | ✗ | `M2` (= M1+M2) — already exists |
| 7 | + Dual head + Composite loss | ✗ | ✓ | ✓ | `M23` (§22) (= M2+M3) — already exists |

Two arms are genuinely new because they isolate a factor **on its own**,
without the color-feature confound the original incremental study baked in:

- **`M2pure`** — dual head alone, trained with `loss_dual_mse` (regression MSE
  + ordinal BCE), reusing the same architecture as `M23` from §22, just a
  different loss and a different `model_id` (so its checkpoints don't
  collide with `M23`'s).
- **`M3pure`** — "composite loss" alone needs a definition when there's no
  ordinal head to feed: it's implemented here as the *huber + ranking* terms
  from `loss_composite`, dropping the ordinal term (there's nothing to apply
  it to without a second head), on the same single-head architecture as `M0`.

Both reuse existing classes/losses — no new architectures were written, and
the shared training engine (`train_one_fold` / `get_preds_and_targets`) got
one small, additive `'M3pure'` branch added above (§10) so a single-head
model can be trained with a custom loss; all other model IDs are unaffected.

This section only depends on cells through §11/§12 (same as §22).

In [ ]:
# ── Dependency check for this section ────────────────────────────────────────
_required_full = [
    'nn', 'torch', 'optim', 'np', 'pd', 'time', 'os',
    'device', 'BATCH_SIZE', 'EPOCHS', 'K_FOLDS', 'SEED', 'IDX_TO_SCORE',
    'M0_Baseline', 'loss_ordinal_bce', 'loss_huber', 'loss_ranking', 'loss_dual_mse',
    'y_raw', 'fold_splits', 'augment_fold', 'compute_metrics',
    'train_one_fold', 'get_preds_and_targets', 'ALL_RESULTS', 'MODEL_LABELS',
]
_missing_full = [n for n in _required_full if n not in globals()]
if _missing_full:
    raise NameError(
        'Missing from kernel: ' + ', '.join(_missing_full) +
        '. Run the base ablation-study cells first (through section 11/12) '
        'in this same kernel before running this section.'
    )
if 'M23_DualHead_CompositeLoss_NoColor' not in globals():
    raise NameError(
        "M23_DualHead_CompositeLoss_NoColor is missing -- run section 22a "
        "first (it's reused here for the 'M2pure' arm)."
    )
for _mid in ('M0', 'M1', 'M2', 'M3'):
    if _mid not in ALL_RESULTS:
        raise NameError(f"ALL_RESULTS['{_mid}'] missing -- run section 11 first.")
if 'M23' not in ALL_RESULTS:
    raise NameError("ALL_RESULTS['M23'] missing -- run section 22b first.")
print('All dependencies for the full-grid section found ✔')

### 23a · `M3pure` loss — composite-style loss with no ordinal head

In [ ]:
def loss_composite_single_head(reg_out, target):
    """'Composite loss' isolated on a single regression head: same huber +
    ranking terms and weights (0.5/0.5) as the full `loss_composite`, minus
    the ordinal-BCE term (there's no ordinal head to apply it to here)."""
    l_hub  = loss_huber(reg_out, target)
    l_rank = loss_ranking(reg_out, target)
    return 0.5 * l_hub + 0.5 * l_rank

print("loss_composite_single_head defined ✔  (0.5·huber + 0.5·rank, no ordinal term)")

### 23b · Train `M2pure` (Dual Head alone, no color) — 5-Fold CV

In [ ]:
print('\n' + '='*70)
print('  M2pure -- Dual Head Alone (No Color, No Composite Loss)'.center(70))
print('='*70)
print('  loss: loss_dual_mse (regression MSE + ordinal BCE)')
print('  ensemble blend: 0.7 x ordinal + 0.3 x regression (same as M2/M3)')

oof_pred_all_2p  = np.full(len(y_raw), np.nan, dtype=np.float32)
oof_true_all_2p  = IDX_TO_SCORE[y_raw]
oof_label_all_2p = y_raw.copy()
fold_hists_2p, fold_metrics_list_2p = [], []

t0 = time.time()
for fold_idx, (tr_idx, va_idx) in enumerate(fold_splits):
    print(f'\n  -- Fold {fold_idx+1}/{K_FOLDS} ----------------------------')
    oof_p, oof_t, oof_l, hist, best_ep = train_one_fold(
        M23_DualHead_CompositeLoss_NoColor, 'M2pure', fold_idx, tr_idx, va_idx,
        loss_fn=loss_dual_mse)

    oof_pred_all_2p[va_idx] = oof_p
    fold_hists_2p.append(hist)
    m = compute_metrics(oof_p, oof_t)
    fold_metrics_list_2p.append({**m, 'fold': fold_idx+1, 'best_epoch': best_ep+1})

elapsed = (time.time() - t0) / 60
oof_metrics_2p = compute_metrics(oof_pred_all_2p, oof_true_all_2p)
print(f'\n  ✔ M2pure done in {elapsed:.1f} min')
print(f'  OOF R²={oof_metrics_2p["r2"]:.4f}  RMSE={oof_metrics_2p["rmse"]:.4f}  '
      f'ρ={oof_metrics_2p["rho"]:.4f}  Acc@0={oof_metrics_2p["acc0"]*100:.1f}%  '
      f'Acc@1={oof_metrics_2p["acc1"]*100:.1f}%')

ALL_RESULTS['M2pure'] = {
    'oof_pred':     oof_pred_all_2p,
    'oof_true':     oof_true_all_2p,
    'oof_label':    oof_label_all_2p,
    'fold_hist':    fold_hists_2p,
    'fold_metrics': fold_metrics_list_2p,
    'oof_metrics':  oof_metrics_2p,
}
print("Stored -> ALL_RESULTS['M2pure']")

### 23c · Train `M3pure` (Composite Loss alone, no color, no dual head) — 5-Fold CV

In [ ]:
print('\n' + '='*70)
print('  M3pure -- Composite Loss Alone (No Color, No Dual Head)'.center(70))
print('='*70)
print('  loss: loss_composite_single_head (0.5 x huber + 0.5 x rank)')
print('  architecture: same single-head backbone as M0 (no ordinal head)')

oof_pred_all_3p  = np.full(len(y_raw), np.nan, dtype=np.float32)
oof_true_all_3p  = IDX_TO_SCORE[y_raw]
oof_label_all_3p = y_raw.copy()
fold_hists_3p, fold_metrics_list_3p = [], []

t0 = time.time()
for fold_idx, (tr_idx, va_idx) in enumerate(fold_splits):
    print(f'\n  -- Fold {fold_idx+1}/{K_FOLDS} ----------------------------')
    oof_p, oof_t, oof_l, hist, best_ep = train_one_fold(
        M0_Baseline, 'M3pure', fold_idx, tr_idx, va_idx,
        loss_fn=loss_composite_single_head)

    oof_pred_all_3p[va_idx] = oof_p
    fold_hists_3p.append(hist)
    m = compute_metrics(oof_p, oof_t)
    fold_metrics_list_3p.append({**m, 'fold': fold_idx+1, 'best_epoch': best_ep+1})

elapsed = (time.time() - t0) / 60
oof_metrics_3p = compute_metrics(oof_pred_all_3p, oof_true_all_3p)
print(f'\n  ✔ M3pure done in {elapsed:.1f} min')
print(f'  OOF R²={oof_metrics_3p["r2"]:.4f}  RMSE={oof_metrics_3p["rmse"]:.4f}  '
      f'ρ={oof_metrics_3p["rho"]:.4f}  Acc@0={oof_metrics_3p["acc0"]*100:.1f}%  '
      f'Acc@1={oof_metrics_3p["acc1"]*100:.1f}%')

ALL_RESULTS['M3pure'] = {
    'oof_pred':     oof_pred_all_3p,
    'oof_true':     oof_true_all_3p,
    'oof_label':    oof_label_all_3p,
    'fold_hist':    fold_hists_3p,
    'fold_metrics': fold_metrics_list_3p,
    'oof_metrics':  oof_metrics_3p,
}
print("Stored -> ALL_RESULTS['M3pure']")

### 23d · Full grid comparison — CV (mean±std, all metrics) + OOF, all 7 arms

In [ ]:
os.makedirs('training', exist_ok=True)

FULL_GRID_LABELS = {
    'M0':     'Baseline (no color, single head, MSE)',
    'M1':     '+ Color Feats',
    'M2pure': '+ Dual Head',
    'M3pure': '+ Composite Loss',
    'M2':     '+ Color Feats + Dual Head             (= M1+M2)',
    'M3':     '+ Color Feats + Dual Head + Composite Loss  (= M1+M2+M3)',
    'M23':    '+ Dual Head + Composite Loss           (= M2+M3)',
}

METRIC_COLS  = ['r2', 'rmse', 'rho', 'tau', 'acc0', 'acc1']
METRIC_NAMES = {'r2': 'R2', 'rmse': 'RMSE', 'rho': 'Spearman rho',
                'tau': 'Kendall tau', 'acc0': 'Acc@0 (%)', 'acc1': 'Acc@1 (%)'}
PCT_METRICS  = {'acc0', 'acc1'}

rows = []
for mid, label in FULL_GRID_LABELS.items():
    if mid not in ALL_RESULTS:
        print(f'  [skip] {label} -- ALL_RESULTS["{mid}"] not found (run its training cell first)')
        continue
    res = ALL_RESULTS[mid]
    m   = res['oof_metrics']
    fm  = pd.DataFrame(res['fold_metrics'])
    row = {'Arm': label}
    for mc in METRIC_COLS:
        scale  = 100 if mc in PCT_METRICS else 1
        fmt    = '.1f' if mc in PCT_METRICS else '.4f'
        oof_v  = m[mc] * scale
        cv_m   = fm[mc].mean() * scale
        cv_s   = fm[mc].std()  * scale
        row[f'{METRIC_NAMES[mc]} (OOF)']          = f'{oof_v:{fmt}}'
        row[f'{METRIC_NAMES[mc]} (CV mean+-std)'] = f'{cv_m:{fmt}}+-{cv_s:{fmt}}'
    rows.append(row)

df_full_grid = pd.DataFrame(rows)

print('\n' + '='*135)
print('  FULL ABLATION GRID -- OOF + 5-Fold CV Metrics (mean +/- std), 7 arms'.center(135))
print('='*135)
print(df_full_grid.to_string(index=False))
print('='*135)

df_full_grid.to_csv('training/comparison_table_full_grid.csv', index=False)
print('\nSaved -> training/comparison_table_full_grid.csv')

# ── Delta vs baseline, for quick reading ─────────────────────────────────────
if 'M0' in ALL_RESULTS:
    m0 = ALL_RESULTS['M0']['oof_metrics']
    print('\nDelta vs M0 (Baseline), OOF:')
    for mid, label in FULL_GRID_LABELS.items():
        if mid == 'M0' or mid not in ALL_RESULTS:
            continue
        m = ALL_RESULTS[mid]['oof_metrics']
        dr2  = m['r2']   - m0['r2']
        da1  = m['acc1'] - m0['acc1']
        print(f'  {label:<58} dR2={dr2:+.4f}   dAcc@1={da1*100:+.1f}%')